# Verifying the 20 m/s / 25 m/s Degraded-Operations Wind Specification

**Specification under test.** The observatory is designed to continue observing in
**degraded operations** up to a **20 m/s sustained wind with 25 m/s gusts**. This notebook
asks one question:

> **Do 25 weeks of LSSTCam on-sky telemetry contradict that specification?**

**The answer is no — and the framing of that "no" matters.** This is a
**bounding / non-contradiction** analysis, not a demonstration of performance at 20 m/s.
Two facts force that framing:

1. **The design point is still never reached — but the sample now gets much closer.** The
   window covers **every LSSTCam science night on sky: 2026-01-07 → 2026-07-13** (116 nights,
   64,411 open-dome science exposures). Maximum *sustained* wind is **18.9 m/s** and maximum
   1-min *gust* is **22.6 m/s**. **Zero exposures** exceed 20 m/s sustained; **zero** exceed
   25 m/s gust. Sustained wind above 14 m/s now occurs on **9 nights** (was 4). Statements
   about 20/25 m/s remain **extrapolations** — but much shorter ones: the V² lever arm to the
   sustained spec is now only **1.12×** (was 1.30×), and **1.75×** to the gust spec (was 2.04×).

   > ⚠️ **Do not confuse the two wind columns.** ConsDB's per-exposure `wind_speed` summary
   > does reach **20.6 m/s** on 2026-05-26, which looks like the spec point is crossed. It is
   > not: that column is a **held/stale summary** (105 distinct values across 828 exposures,
   > the same 20.5998 repeating), whereas the analysis metric `efd_wind_speed` is a fresh
   > 1-min EFD resample (605 distinct values, max 18.9 m/s on that night). They agree overall
   > (r = 0.975, median ratio 1.002) but disagree in the tail that matters most. All results
   > here use the EFD metric.

2. **Two anchors now show a real within-night wind response; the tracking one is the story.**
   Naive pooled fits `metric = a + k·V²` give positive slopes for everything, but most of that
   is **between-night (seasonal/synoptic) confounding** — the estimator artifact that
   invalidated the earlier dome-seeing conclusion here (see
   [`Dome_Seeing_Model_Construction.ipynb`](Dome_Seeing_Model_Construction.ipynb)). Under
   night fixed effects with cluster-robust SEs on the full 116-night window:
   - **Mount tracking jitter — real signal, t ≈ +5.6** (was +4.6 on 73 nights). It projects to
     **71% of the 0.047″ allocation at 20 m/s (CONSISTENT)** but its 95% upper bound reaches
     **U = 1.05 at the 25 m/s gust condition** — i.e. *marginally* not certifiable there. The
     central estimate stays inside budget; only the pessimistic bound crosses.
   - **M1M3 balance moment — now detectable, t ≈ +2.5**, but it has no official tolerance in
     telemetry units, so it is reported **NO-BUDGET** rather than pass/fail (see below).
   - **VMS accel, guider RMS — no detectable response** (|t| < 1). The mirror is not
     wind-limited.

   > **Why the force anchor cannot fail the spec.** Its budget is "+100% over calm", a
   > *screening* choice, not an engineering tolerance. Since wind load ∝ V², that line is
   > crossed at ≈3·√2 = **4.2 m/s by the V² law alone** — exceeding it at 20 m/s restates the
   > physics rather than revealing a fault. Only the two tracking anchors carry absolute
   > budgets (0.047″), so only they can genuinely contradict the spec.

So the deliverable is: a **measured, in-budget response** for sustained wind; a **marginal
non-certification at the gust condition**; and conservative **upper bounds** elsewhere — the
95% cluster-robust limit `k_within + 1.96·SE` propagated to 20 and 25 m/s.

## What "confirmation" can and cannot mean here

| | Claim | Status |
|---|---|---|
| ✅ | Telemetry is **consistent with** the 20/25 m/s spec | **Supported** — no anchor projects past budget |
| ✅ | Tracking jitter at 20 m/s is **measured in-budget** | **Supported** — real response (t=+5.6) at ~71% of the 0.047″ allocation |
| ⚠️ | Tracking jitter at the **25 m/s gust** is certifiable | **No** — 95% bound reaches U=1.05, marginally over; central estimate is inside |
| ✅ | Wind-driven mechanical degradation is **bounded small** | **Supported** — 95% bounds propagated below |
| ✅ | Dome interior wind coupling is **≈10%** of outside speed | **Supported** (§15b) |
| ❌ | Performance at 20 m/s is **measured** | **Not supported** — 0 exposures there |
| ❌ | A budget **crossing** was detected | **Not supported** — nothing reaches budget in or beyond the observed range |
| ⚠️ | Force / VMS / guider bounds **certify** the spec | **Underpowered** — the 17.5 m/s ceiling leaves CIs too wide (esp. force) |

> **Read the bounds as one-sided.** A bound that clears budget is evidence of
> *non-contradiction*. A bound that does **not** clear budget is **not** evidence of
> failure — with a 17.5 m/s data ceiling, a V²-extrapolation to 25 m/s inflates the slope
> uncertainty by (25/17.5)² ≈ 2×, so a loose bound reflects **limited statistical power**,
> not observed degradation. We label these `underpowered` rather than `FAIL`.

## Anchors and budgets

Four **mechanical** anchors. The image-quality anchors (PSF FWHM, donut blur) that appeared
in earlier revisions have been **removed**: their apparent wind response was a
between-night artifact that vanishes within nights (r(V², PSF) = +0.02), so they cannot
support an operational limit. The dome-seeing pathway is treated properly in
[`Dome_Seeing_Model_Construction.ipynb`](Dome_Seeing_Model_Construction.ipynb), where the
driver is shown to be **mirror–air ΔT (≈0.10″/K)**, with wind only an upper limit.

| # | Anchor | Source | Budget | Rationale |
|---|--------|--------|--------|-----------|
| a | **M1M3 balance moment** (mx,my) | EFD `MTM1M3.appliedBalanceForces` | +100% over calm | Wind load on the mirror; force-balance loop response, gravity removed |
| b | **Mount tracking jitter** | ConsDB `mount_jitter_rms` | **0.047″** | Wind-buffeting jitter allocation |
| c | **M1M3 VMS accel RMS** | EFD `MTVMS.data` (salIndex=1) | +100% over calm | Dynamic/oscillatory response (IMS std is floor-limited) |
| e | **Guider RMS motion** | ConsDB `visit1_quicklook` (mas→arcsec) | **0.047″** | Direct in-exposure image motion, seeing-independent |

> **Tracking-jitter budget.** Anchors (b) and (e) use the **wind-buffeting allocation
> (0.047″)**, *not* the 0.01″ total tracking-jitter design requirement. The 0.01″ figure is
> the full error budget across all sources (servo, encoder, thermal, wind); since this
> analysis isolates the wind contribution, 0.047″ is the correct comparison.
>
> **Force anchor.** We anchor on the balance **moment**, not raw hardpoint forces: the
> force-balance loop drives hardpoint loads to ~zero, so HP forces are the loop *error
> residual* (slew-transient-dominated). Per-exposure statistics skip the first 5 s
> (slew-settle) and use the native ~50 Hz rate.

## Gust treatment

No exposure reaches 25 m/s gust, so the gust case is evaluated by **scaling the sustained
result by the spec's own gust factor 25/20 = 1.25** (§20). As an independent consistency
check, the *observed* gust factor (`efd_wind_speed_max / efd_wind_speed`) has median ≈1.13
and p90 ≈1.29 — it **brackets 1.25**, so the spec's sustained/gust pairing is internally
consistent with Cerro Pachón statistics (§10b).

## Structure

- **Layer A (§10–§15b):** measured telemetry per science exposure, incl. spec-reach coverage
  (§10b) and the inside/outside dome wind coupling (§15b).
- **Layer B (§16–§18):** within-night attribution (§16 — the estimator that decides whether
  any wind signal is real) and the first-principles aerodynamic model (§17–§18), which
  carries the physics of the extrapolation to the design point.
- **Layer C (§19–§22):** budgets, the **spec-verification verdict** at 20/25 m/s, sensitivity,
  and recommendations.

**Relative-wind convention:** `relative_wind = wrap180(wind_direction − dome_azimuth)`;
**0° = INTO the wind**, **±180° = AWAY**. `into_wind = |Δ|<45°`, `away_wind = |Δ|>135°`.

> ⚠️ **Selection bias is central to the verdict.** Every row is an open-dome frame the
> observatory *chose* to take: operators avoid high wind and point away from it when gusty
> (§10). The high-wind/into-wind corner is self-censored, so measured curves alone
> **understate** true risk — which is exactly why the verdict rests on conservative
> statistical bounds plus the telemetry-validated aerodynamic model, not on raw
> extrapolation of observed means.


## §1 — Imports

In [ ]:
import os
import pathlib
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import sqlalchemy

from scipy import signal, stats
from scipy import (
    stats as spstats,
)  # alias: `stats` is reused as a local var in the mirror-stats join

from astropy.time import Time, TimeDelta
import astropy.units as u

from lsst_efd_client import EfdClient

warnings.filterwarnings("ignore", category=RuntimeWarning)  # nan-slices in binned stats
%matplotlib inline
pd.set_option("display.max_columns", 60)

## §2 — Configuration

Single control cell. The `VALIDATE_WINDOW` toggle switches between a short recent window
(fast, for building/validating) and the full 25-week window. `day_obs_start/end` and the
cache filename are derived **after** the toggle so ConsDB, EFD, and the cache stay
mutually consistent — the validate and full runs write to distinct cache files keyed by
`day_obs`, so they never collide.

In [ ]:
# ── ConsDB / PostgreSQL (verbatim from ConsDB_EFD_to_PSF_Effects_Diagnosis.ipynb) ──
PGPASS_FILE = os.path.expanduser("~/.lsst/postgres-credentials.txt")
CONSDB_HOST = "usdf-summitdb-logical-replica-svc.sdf.slac.stanford.edu"
CONSDB_DB = "exposurelog"
CONSDB_USER = "usdf"
SCHEMA = "cdb_lsstcam"


def load_pgpass(path, host, database, user):
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = line.split(":")
            if len(parts) < 5:
                continue
            h, db, u_ = parts[0], parts[2], parts[3]
            pwd = ":".join(parts[4:])
            if h == host and db == database and u_ == user:
                return pwd
    raise ValueError(f"No credentials for {user}@{host}/{database}")


db_pass = load_pgpass(PGPASS_FILE, CONSDB_HOST, CONSDB_DB, CONSDB_USER)
engine = sqlalchemy.create_engine(
    f"postgresql+psycopg2://{CONSDB_USER}:{db_pass}@{CONSDB_HOST}/{CONSDB_DB}",
    connect_args={"connect_timeout": 30},
)


def consdb_query(sql):
    with engine.connect() as conn:
        return pd.read_sql_query(sqlalchemy.text(sql), conn)


print("ConsDB schemas:")
display(
    consdb_query(
        "SELECT schema_name FROM information_schema.schemata "
        "WHERE schema_name LIKE 'cdb_%' ORDER BY 1"
    )
)

In [ ]:
# ── Time window ──────────────────────────────────────────────────────────────
# FULL ON-SKY WINDOW: 2026-01-07 → 2026-07-14, covering every LSSTCam science
# night through the final night on sky, day_obs=2026-07-13 (2026-07-14 has
# calibrations only: flats/darks/biases, which the open-dome guard in §9 removes).
#
# NOTE: an earlier revision stopped at 2026-04-27 on the belief that the survey
# paused there. That was WRONG — 2026-04-28..2026-07-13 are normal survey nights
# (DD fields, pair_15 visits). The extension matters for the spec question: the
# added nights contain the windiest science data of the campaign, reaching
# ~20.6 m/s sustained on 2026-05-26 and finally CROSSING the 20 m/s design point
# that the 73-night window never reached (its max was 17.5 m/s).
ANCHOR_TO_LAST_DATA = False


def _last_science_day():
    r = consdb_query(
        f"SELECT max(day_obs) AS mx FROM {SCHEMA}.exposure WHERE img_type='science'"
    )
    return int(r.iloc[0, 0])


if ANCHOR_TO_LAST_DATA:
    _mx = _last_science_day()
    _s = str(_mx)
    t_end = Time(f"{_s[:4]}-{_s[4:6]}-{_s[6:8]}T12:00:00", scale="utc") + TimeDelta(
        1 * u.day
    )
    print(f"anchored t_end to last science day_obs={_mx}")
else:
    t_end = Time("2026-07-14T12:00:00", scale="utc")

# Literal start date — NOT derived from a week count, so day_obs_start is exact
# and the cache filename can never drift on float rounding.
FULL_START = Time("2026-01-07T12:00:00", scale="utc")

# VALIDATE_WINDOW=True → short recent window to build & sanity-check the notebook.
# Flip to False for the full 25-week window (fetched via the standalone cache
# builder build_full_cache.py; the notebook then just loads that cache).
VALIDATE_WINDOW = False
VALIDATE_WEEKS = 2

if VALIDATE_WINDOW:
    t_start = t_end - TimeDelta(VALIDATE_WEEKS * u.week)
else:
    t_start = FULL_START
_weeks = (t_end - t_start).to(u.week).value

day_obs_start = int(t_start.strftime("%Y%m%d"))
day_obs_end = int(t_end.strftime("%Y%m%d"))

print(f"{'VALIDATE' if VALIDATE_WINDOW else 'FULL'} window: {_weeks:.1f} weeks")
print(f"  {t_start.iso}  →  {t_end.iso}")
print(f"  day_obs {day_obs_start} → {day_obs_end}")

# ── Parquet cache (filename encodes the window) ──────────────────────────────
CACHE_DIR = pathlib.Path("../data")
CACHE_FILE = CACHE_DIR / f"wind_loading_{day_obs_start}_{day_obs_end}.parquet"
PSD_CACHE_FILE = CACHE_DIR / f"wind_loading_psd_{day_obs_start}_{day_obs_end}.parquet"
FORCE_REFETCH = False
INCREMENTAL_REFETCH = False
print(f"\ncache: {CACHE_FILE}")

In [ ]:
# ── EFD topics / indices ─────────────────────────────────────────────────────
EFD_ALIAS = "usdf_efd"

flow_topic = "lsst.sal.ESS.airFlow"  # outside wind
turb_topic = "lsst.sal.ESS.airTurbulence"  # inside dome turbulence
dome_topic = "lsst.sal.MTDome.azimuth"  # dome azimuth
shutter_topic = "lsst.sal.MTDome.apertureShutter"
pressure_topic = "lsst.sal.ESS.pressure"  # for air density
particle_topic = "lsst.sal.ESS.particleMeasurements"  # dust
weather_index = 301  # outside weather station
pressure_index = 301
shutter_threshold = 95.0  # % open

# Mount actual pointing (field name confirmed live in §4)
mount_el_topic = "lsst.sal.MTMount.elevation"
mount_az_topic = "lsst.sal.MTMount.azimuth"
MOUNT_POS_FIELD = "actualPosition"  # verified/overridden in §4

ess_sensors = {
    110: "TMA Platform",
    123: "Top Ring -X/-Y",
    124: "Top Ring +X/-Y",
    125: "Top Ring +X/+Y",
    126: "Top Ring -X/+Y",
}
# saturation clip (m/s): top-ring sensors saturate ~8.6 m/s; TMA platform unclipped
speed_clip = {110: None, 123: 8.6, 124: 8.6, 125: 8.6, 126: 8.6}

particle_indices = {127: "Dome PM A", 128: "Dome PM B", 129: "Dome PM C"}

# High-rate mirror / force telemetry (per-exposure mean/std + spectral)
mirror_topics = {
    "m1m3ims": {
        "topic": "lsst.sal.MTM1M3.imsData",
        "fields": [
            "xPosition",
            "yPosition",
            "zPosition",
            "xRotation",
            "yRotation",
            "zRotation",
        ],
    },
    "m1m3hp": {
        "topic": "lsst.sal.MTM1M3.hardpointActuatorData",
        "fields": [f"measuredForce{i}" for i in range(6)],
    },
    # appliedBalanceForces: force-balance loop's aggregate force/moment. The balance
    # MOMENTS (mx,my) are the physical wind-load signal (gravity is carried by the
    # static support); HP forces are the loop error residual (≈0 steady-state) and are
    # kept only for diagnostic comparison. Balance mx,my give the moment directly.
    "m1m3bal": {
        "topic": "lsst.sal.MTM1M3.appliedBalanceForces",
        "fields": ["fx", "fy", "fz", "mx", "my", "mz"],
    },
    "m2pos": {
        "topic": "lsst.sal.MTM2.positionIMS",
        "fields": ["x", "y", "z", "xRot", "yRot", "zRot"],
    },
}

# Skip the first N s of each exposure window: the mount has just finished slewing and
# the M1M3 support is still ringing down — those forces are slew-settle transients, not
# wind loading (verified: dynamic RMS ~halves when the leading 5 s is dropped).
SETTLE_SKIP_S = 5.0

In [ ]:
# ── Binning idioms (match sibling dome/wind notebooks) ───────────────────────
wind_speed_bins = [0, 3, 6, 9, 12, np.inf]
wind_speed_labels = ["0-3", "3-6", "6-9", "9-12", ">12"]
wind_speed_colors = [
    "steelblue",
    "mediumseagreen",
    "goldenrod",
    "tomato",
    "mediumpurple",
]

rel_wind_bins = np.arange(-180, 181, 15)  # 15° relative-wind bins
speed_fit_bins = np.arange(0, 20.1, 1.0)  # 1 m/s bins for limit fits

INTO_WIND_MAX = 45.0  # |rel_wind| < this  → pointed into the wind
AWAY_WIND_MIN = 135.0  # |rel_wind| > this  → pointed away from the wind

# ── Aerodynamic model constants — ALL ASSUMPTIONS (stress-tested in §21) ─────
R_AIR = 287.1  # J/(kg·K) dry-air gas constant
RHO_REF = 0.80  # kg/m³ fallback air density at Cerro Pachón (~2650 m, ~5°C)
A_M1M3 = 55.0  # m²  face-on projected area of the 8.4 m primary  [ASSUMPTION]
CD_MIRROR = 1.2  # flat-plate normal drag coefficient               [ASSUMPTION]
CD_RANGE = (1.1, 1.3)
LEVER_ARM_M = 2.0  # m   elevation-axis → M1M3 centre-of-pressure      [ASSUMPTION]
SHUTTER_OPEN_AREA = 200.0  # m²  full-open aperture (no LWS)
DOME_DIAMETER = 30.0  # m

# ── Operational budgets (anchor thresholds) — set in §19, placeholder here ───
BUDGETS = {}  # populated in §19


# ── Plot helpers ─────────────────────────────────────────────────────────────
def wrap180(a):
    """Wrap angle(s) to [-180, 180] degrees."""
    return ((np.asarray(a, dtype=float) + 180.0) % 360.0) - 180.0


def to_epoch(ts):
    """pandas DatetimeIndex → seconds since epoch (for np.interp)."""
    return ts.astype(np.int64) / 1e9


def circular_mean_deg(x):
    x_rad = np.deg2rad(pd.Series(x).dropna())
    if len(x_rad) == 0:
        return np.nan
    return np.rad2deg(np.arctan2(np.sin(x_rad).mean(), np.cos(x_rad).mean())) % 360


# mount jitter / image-degradation physical clip [arcsec]
MOUNT_JITTER_MIN = 0.0
MOUNT_JITTER_MAX = 1.0

print("config loaded")

## §3 — Cache short-circuit

If a cache for this exact window exists (and `FORCE_REFETCH` is off), load it and set
`_from_cache=True`. Every fetch/compute cell below is guarded by `if not _from_cache:`,
so re-runs — and the eventual full 25-week run once its cache is built — are fast.

In [ ]:
_from_cache = False
if not FORCE_REFETCH and CACHE_FILE.exists():
    df_joined = pd.read_parquet(CACHE_FILE)
    _from_cache = True
    print(f"✓ Loaded {len(df_joined):,} rows from cache: {CACHE_FILE}")
    print(f"  {df_joined.index.min()} → {df_joined.index.max()}")
elif FORCE_REFETCH:
    print("FORCE_REFETCH=True — bypassing cache")
else:
    print(f"No cache at {CACHE_FILE} — will fetch fresh data")

## §4 — Schema & live-data probes  *(verification)*

Concentrates all "needs live verification" items before the expensive fetch:
- ConsDB `img_type` counts, date range, and quicklook coverage in-window.
- Guider RMS/drift column names in `visit1_quicklook` (via `information_schema`).
- EFD topic/field probes: `MTMount.elevation` field name, M1M3 IMS/HP **sample rate**
  (sets the §13b PSD Nyquist band), M2, `particleMeasurements` 127/128/129 (may be
  empty — dust sensors are intermittently deployed), `ESS.pressure` 301.

Prints a go/no-go checklist. Safe to run even when `_from_cache=True` (read-only).

In [ ]:
# ConsDB coverage
print("img_type counts:")
display(
    consdb_query(
        f"""
    SELECT img_type, count(*) AS n FROM {SCHEMA}.exposure
    GROUP BY img_type ORDER BY n DESC LIMIT 8
"""
    )
)
print("science + quicklook coverage in-window:")
display(
    consdb_query(
        f"""
    SELECT
      count(*) AS n_science,
      count(q.visit_id) AS n_visit1_quicklook,
      count(m.day_obs)  AS n_exposure_quicklook
    FROM {SCHEMA}.exposure e
    LEFT JOIN {SCHEMA}.visit1_quicklook  q ON q.day_obs=e.day_obs AND q.seq_num=e.seq_num
    LEFT JOIN {SCHEMA}.exposure_quicklook m ON m.day_obs=e.day_obs AND m.seq_num=e.seq_num
    WHERE e.img_type='science' AND e.day_obs BETWEEN {day_obs_start} AND {day_obs_end}
"""
    )
)

In [ ]:
# Discover guider RMS / drift columns in visit1_quicklook
guider_cols_df = consdb_query(
    f"""
    SELECT column_name, data_type FROM information_schema.columns
    WHERE table_schema='{SCHEMA}' AND table_name='visit1_quicklook'
      AND column_name ILIKE '%guider%'
    ORDER BY ordinal_position
"""
)
print(f"{len(guider_cols_df)} guider columns in visit1_quicklook:")
display(guider_cols_df)

# Candidate guider columns we want (intersect with what actually exists)
GUIDER_WANT = [
    "guider_altitude_drift",
    "guider_azimuth_drift",
    "guider_magnitude_drift",
    "guider_altitude_rms_detrended",
    "guider_azimuth_rms_detrended",
    "guider_magnitude_rms_detrended",
    "guider_focalplane_theta_drift",
    "guider_focalplane_theta_rms_detrended",
]
_avail = set(guider_cols_df["column_name"])
GUIDER_COLS = [c for c in GUIDER_WANT if c in _avail]
print(f"\nUsing guider columns: {GUIDER_COLS}")
if not GUIDER_COLS:
    print("⚠ no expected guider columns present — anchor (e) will be unavailable")

In [ ]:
# EFD probes — use select_time_series directly (schema registry not reachable
# from SDF; get_schema() fails but data fetches work). Short 2-hour window.
efd_client = EfdClient(EFD_ALIAS)
print("Connected to EFD\n")

t_probe_end = t_start + TimeDelta(2 * u.hour)


async def probe(topic, fields, index=None):
    try:
        df = await efd_client.select_time_series(
            topic, fields=fields, start=t_start, end=t_probe_end, index=index
        )
        dt = (
            np.median(np.diff(df.index.astype(np.int64) / 1e9))
            if len(df) > 3
            else np.nan
        )
        rate = 1.0 / dt if (dt == dt and dt > 0) else np.nan
        print(
            f"  {topic}{f' [idx={index}]' if index is not None else ''}: "
            f"rows/2h={len(df)}  ~{rate:.1f} Hz  cols={list(df.columns)[:6]}"
        )
        return df, rate
    except Exception as e:
        print(f"  {topic}  → {str(e)[:90]}")
        return None, np.nan


print("── Mount pointing ──")
_df_el, _ = await probe(mount_el_topic, [MOUNT_POS_FIELD])
if _df_el is None:
    # resolve the field name if the default is wrong
    for cand in ("positionActual", "actualPositionTimestamp"):
        _df_el, _ = await probe(mount_el_topic, [cand])
        if _df_el is not None:
            MOUNT_POS_FIELD = cand
            print(f"    → MOUNT_POS_FIELD resolved to '{MOUNT_POS_FIELD}'")
            break

print("\n── High-rate mirror / force (sets §13b PSD band) ──")
_ims_df, IMS_RATE = await probe(
    mirror_topics["m1m3ims"]["topic"], ["xPosition", "zPosition"]
)
_hp_df, HP_RATE = await probe(mirror_topics["m1m3hp"]["topic"], ["measuredForce0"])
_m2_df, M2_RATE = await probe(mirror_topics["m2pos"]["topic"], ["x", "z"])

print("\n── Environment / dust ──")
_p_df, _ = await probe(pressure_topic, ["pressureItem0"], index=pressure_index)
_w_df, _ = await probe(flow_topic, ["speed", "direction"], index=weather_index)
PARTICLE_OK = {}
for idx, lab in particle_indices.items():
    dfp, _ = await probe(particle_topic, ["numberConcentration0"], index=idx)
    PARTICLE_OK[idx] = dfp is not None and len(dfp) > 0

print("\n──────── GO/NO-GO CHECKLIST ────────")
print(f"  guider anchor (e)     : {'OK' if GUIDER_COLS else 'MISSING'}")
print(
    f"  mount pos field       : {MOUNT_POS_FIELD} ({'OK' if _df_el is not None else 'CHECK'})"
)
print(f"  IMS / HP / M2 rate Hz : {IMS_RATE:.0f} / {HP_RATE:.0f} / {M2_RATE:.0f}")
print(
    f"  pressure (ρ)          : {'OK' if _p_df is not None and len(_p_df) else 'FALLBACK RHO_REF'}"
)
print(
    f"  outside wind          : {'OK' if _w_df is not None and len(_w_df) else 'MISSING'}"
)
print(f"  particle (dust)       : {PARTICLE_OK}")

## §5 — Fetch science exposures + quicklook (ConsDB)

One row per science exposure: pointing, ConsDB wind summary, PSF/AOS, mount jitter, and
the guider RMS/drift columns (anchor e). Guider columns are injected dynamically from the
`GUIDER_COLS` list resolved in §4, so the query adapts to what the schema actually has.

In [ ]:
if not _from_cache:
    _guider_sql = (
        ",\n            " + ",\n            ".join(f"q.{c}" for c in GUIDER_COLS)
        if GUIDER_COLS
        else ""
    )
    df_exp = consdb_query(
        f"""
        SELECT
            e.exposure_id, e.day_obs, e.seq_num, e.obs_start, e.obs_start_mjd, e.exp_time,
            e.airmass, e.s_ra, e.s_dec, e.azimuth, e.altitude, e.sky_rotation,
            e.wind_speed, e.wind_dir, e.dimm_seeing, e.air_temp, e.humidity,
            e.band, e.physical_filter, e.target_name, e.img_type,
            e.vignette, e.can_see_sky, e.scheduler_note,
            q.psf_sigma_median, q.seeing_zenith_500nm_median, q.eff_time_median,
            q.eff_time_zero_point_scale_median, q.sky_bg_median, q.zero_point_median,
            q.donut_blur_fwhm, q.aos_fwhm,
            q.z4, q.z5, q.z6, q.z7, q.z8, q.z9, q.z10, q.z11,
            q.psf_ixx_median, q.psf_iyy_median, q.psf_ixy_median{_guider_sql},
            m.mount_motion_image_degradation, m.mount_motion_image_degradation_az,
            m.mount_motion_image_degradation_el, m.mount_motion_image_degradation_rot,
            m.mount_jitter_rms, m.mount_jitter_rms_az,
            m.mount_jitter_rms_el, m.mount_jitter_rms_rot
        FROM {SCHEMA}.exposure e
        LEFT JOIN {SCHEMA}.visit1_quicklook  q ON q.day_obs=e.day_obs AND q.seq_num=e.seq_num
        LEFT JOIN {SCHEMA}.exposure_quicklook m ON m.day_obs=e.day_obs AND m.seq_num=e.seq_num
        WHERE e.day_obs BETWEEN {day_obs_start} AND {day_obs_end}
          AND e.img_type='science'
        ORDER BY e.obs_start_mjd
    """
    )
    df_exp["obs_start_utc"] = pd.to_datetime(df_exp["obs_start"], utc=True)
    df_exp = df_exp.sort_values("obs_start_utc").set_index("obs_start_utc")
    print(f"Science exposures : {len(df_exp)}")
    print(f"  date range      : {df_exp.index.min()} → {df_exp.index.max()}")
    print(f"  with PSF sigma  : {df_exp['psf_sigma_median'].notna().sum()}")
    print(f"  with AOS FWHM   : {df_exp['aos_fwhm'].notna().sum()}")
    print(f"  with mount jit  : {df_exp['mount_jitter_rms'].notna().sum()}")
    if GUIDER_COLS:
        print(
            f"  with guider RMS : {df_exp[GUIDER_COLS[0]].notna().sum()} ({GUIDER_COLS[0]})"
        )
    display(df_exp.head(3))

## §6 — Fetch EFD telemetry  *(async, daily-chunked)*

All raw series pulled once. `fetch_chunked` splits into daily pieces to stay under the
EFD row limit — **mandatory** for the high-rate M1M3 IMS/hardpoint topics. Empty results
(e.g. dust sensors) are tolerated gracefully.

In [ ]:
if not _from_cache:

    async def fetch_chunked(
        topic, fields, t0, t1, index=None, chunk=TimeDelta(1 * u.day)
    ):
        chunks, t_lo = [], t0
        while t_lo < t1:
            t_hi = min(t_lo + chunk, t1)
            try:
                df = await efd_client.select_time_series(
                    topic, fields=fields, start=t_lo, end=t_hi, index=index
                )
                if not df.empty:
                    chunks.append(df)
            except Exception as e:
                print(f"    chunk {t_lo.iso[:10]} {topic.split('.')[-1]} error: {e}")
            t_lo = t_hi
        return pd.concat(chunks).sort_index() if chunks else pd.DataFrame()

In [ ]:
if not _from_cache:
    # ── Low-rate environment / dome ──────────────────────────────────────────
    df_wind_raw = await fetch_chunked(
        flow_topic, ["direction", "speed"], t_start, t_end, index=weather_index
    )
    print(f"airFlow (301)      rows: {len(df_wind_raw):,}")
    df_dome = await fetch_chunked(dome_topic, ["positionActual"], t_start, t_end)
    print(f"dome azimuth       rows: {len(df_dome):,}")
    df_shutter = await fetch_chunked(
        shutter_topic, ["positionActual0", "positionActual1"], t_start, t_end
    )
    print(f"aperture shutter   rows: {len(df_shutter):,}")

    # mount actual pointing (elevation & azimuth) — for data-driven projected area
    df_mount_el = await fetch_chunked(mount_el_topic, [MOUNT_POS_FIELD], t_start, t_end)
    df_mount_az = await fetch_chunked(mount_az_topic, [MOUNT_POS_FIELD], t_start, t_end)
    print(f"mount el / az      rows: {len(df_mount_el):,} / {len(df_mount_az):,}")

    # pressure for air density
    df_pressure = await fetch_chunked(
        pressure_topic, ["pressureItem0"], t_start, t_end, index=pressure_index
    )
    print(f"pressure (301)     rows: {len(df_pressure):,}")

In [ ]:
if not _from_cache:
    # ── Inside turbulence sensors ────────────────────────────────────────────
    ess_data = {}
    for idx, label in ess_sensors.items():
        df = await fetch_chunked(
            turb_topic,
            ["speedMagnitude", "sonicTemperatureStdDev"],
            t_start,
            t_end,
            index=idx,
        )
        ess_data[idx] = df
        print(f"  turb [{idx}] {label:16s} rows={len(df):,}")

    # ── Dust / particulates (may be empty) ───────────────────────────────────
    particle_data = {}
    _pm_fields = [f"numberConcentration{i}" for i in range(5)] + [
        f"matterConcentration{i}" for i in range(5)
    ]
    for idx, label in particle_indices.items():
        try:
            df = await fetch_chunked(
                particle_topic, _pm_fields, t_start, t_end, index=idx
            )
        except Exception as e:
            df = pd.DataFrame()
            print(f"  pm [{idx}] error: {e}")
        particle_data[idx] = df
        print(f"  pm   [{idx}] {label:16s} rows={len(df):,}")

In [ ]:
if not _from_cache:
    # ── High-rate mirror / force telemetry (daily chunks; heavy) ─────────────
    mirror_data = {}
    for key, cfg in mirror_topics.items():
        print(f"[{key}] fetching {cfg['topic']} ...", flush=True)
        df = await fetch_chunked(cfg["topic"], cfg["fields"], t_start, t_end)
        mirror_data[key] = df if not df.empty else None
        print(f"  → {len(df):,} rows" if not df.empty else "  → no data")

## §7 — Derived per-exposure environmental context

Attach the split-axis variables and physical drivers to each exposure: dome azimuth,
shutter state, mount actual el/az, matched outside wind (1-min `merge_asof`), the
**relative-wind angle**, into/away flags, and air density `ρ = p/(R_air·T)`.

In [ ]:
if not _from_cache:
    exp_midpt = df_exp.index + pd.to_timedelta(df_exp["exp_time"] / 2, unit="s")
    exp_epoch = to_epoch(exp_midpt)

    def _interp_to_exp(df_src, col):
        if df_src is None or df_src.empty:
            return np.full(len(df_exp), np.nan)
        return np.interp(exp_epoch, to_epoch(df_src.index), df_src[col].values)

    # dome azimuth, shutter, mount pointing at exposure midpoint
    df_exp["dome_azimuth_efd"] = _interp_to_exp(df_dome, "positionActual")
    _L = _interp_to_exp(df_shutter, "positionActual0")
    _R = _interp_to_exp(df_shutter, "positionActual1")
    df_exp["shutter_open"] = (_L > shutter_threshold) & (_R > shutter_threshold)
    df_exp["mount_el_actual"] = _interp_to_exp(df_mount_el, MOUNT_POS_FIELD)
    df_exp["mount_az_actual"] = _interp_to_exp(df_mount_az, MOUNT_POS_FIELD)
    print(
        f"shutter-open science exposures: {df_exp['shutter_open'].sum()} / {len(df_exp)}"
    )

In [ ]:
if not _from_cache:
    # ── Outside wind: 1-min resample → merge_asof (±3 min) ───────────────────
    df_wind_1m = df_wind_raw.resample("60s").agg(
        efd_wind_speed=("speed", "mean"),
        efd_wind_speed_max=("speed", "max"),
        efd_wind_speed_std=("speed", "std"),
        efd_wind_dir=("direction", circular_mean_deg),
    )
    df_wind_1m.index.name = "time"
    df_joined = pd.merge_asof(
        df_exp.reset_index()
        .rename(columns={"obs_start_utc": "time"})
        .sort_values("time"),
        df_wind_1m.reset_index().sort_values("time"),
        on="time",
        direction="nearest",
        tolerance=pd.Timedelta("3min"),
    ).set_index("time")
    print(
        f"wind-matched exposures: {df_joined['efd_wind_speed'].notna().sum()} / {len(df_joined)}"
    )

In [ ]:
if not _from_cache:
    # ── Relative wind, into/away split, air density ──────────────────────────
    df_joined["relative_wind"] = wrap180(
        df_joined["efd_wind_dir"] - df_joined["dome_azimuth_efd"]
    )
    df_joined["abs_relative_wind"] = df_joined["relative_wind"].abs()
    df_joined["into_wind"] = df_joined["abs_relative_wind"] < INTO_WIND_MAX
    df_joined["away_wind"] = df_joined["abs_relative_wind"] > AWAY_WIND_MIN

    # air density from pressure (Pa) & air_temp (°C → K); fallback RHO_REF
    p_pa = _interp_to_exp(df_pressure, "pressureItem0")
    # some ESS pressure feeds report hPa/mbar; normalise to Pa if magnitude looks like hPa
    if np.nanmedian(p_pa) < 2000:
        p_pa = p_pa * 100.0
    T_k = df_joined["air_temp"].values + 273.15
    rho = p_pa / (R_AIR * T_k)
    df_joined["rho"] = np.where(
        np.isfinite(rho) & (rho > 0.3) & (rho < 1.5), rho, RHO_REF
    )

    # binning helpers
    df_joined["wind_speed_bin"] = pd.cut(
        df_joined["efd_wind_speed"], bins=wind_speed_bins, labels=wind_speed_labels
    )
    df_joined["rel_wind_bin"] = pd.cut(df_joined["relative_wind"], bins=rel_wind_bins)
    print(
        f"into-wind: {df_joined['into_wind'].sum()}  |  "
        f"away-wind: {df_joined['away_wind'].sum()}  |  "
        f"crosswind: {(~df_joined['into_wind'] & ~df_joined['away_wind']).sum()}"
    )
    print(f"median air density ρ = {df_joined['rho'].median():.3f} kg/m³")

In [ ]:
if not _from_cache:
    # ── Inside turbulence: per-sensor 1-s resample + saturation clip, merge_asof
    for idx, label in ess_sensors.items():
        df = ess_data.get(idx)
        if df is None or df.empty:
            continue
        clip = speed_clip.get(idx)
        if clip is not None:
            df = df[df["speedMagnitude"] < clip]
        df_1s = (
            df[["speedMagnitude", "sonicTemperatureStdDev"]]
            .resample("1s")
            .mean()
            .rename(
                columns={
                    "speedMagnitude": f"turb_speed_{idx}",
                    "sonicTemperatureStdDev": f"turb_sonic_{idx}",
                }
            )
        )
        df_1s.index.name = "time"
        df_joined = pd.merge_asof(
            df_joined.reset_index().sort_values("time"),
            df_1s.reset_index().sort_values("time"),
            on="time",
            direction="nearest",
            tolerance=pd.Timedelta("5s"),
        ).set_index("time")
    # top-ring mean inside-turbulence intensity (a wind/thermal confound covariate)
    _tr = [
        f"turb_speed_{i}"
        for i in (123, 124, 125, 126)
        if f"turb_speed_{i}" in df_joined
    ]
    if _tr:
        df_joined["inside_turb_speed"] = df_joined[_tr].mean(axis=1)
    # ── Dust: per-exposure particle_total_count (mean across dome sensors) ────
    pm_series = []
    for _idx, _dfp in particle_data.items() if "particle_data" in dir() else []:
        if _dfp is None or _dfp.empty:
            continue
        _nc = [c for c in _dfp.columns if c.startswith("numberConcentration")]
        if not _nc:
            continue
        _s = _dfp[_nc].sum(axis=1)
        if _s.index.tzinfo is None:
            _s.index = _s.index.tz_localize("UTC")
        pm_series.append(_s.resample("60s").mean().rename(f"pm_{_idx}"))
    if pm_series:
        _pm = pd.concat(pm_series, axis=1).mean(axis=1).rename("particle_total_count")
        _pm.index.name = "time"
        df_joined = pd.merge_asof(
            df_joined.reset_index().sort_values("time"),
            _pm.reset_index().sort_values("time"),
            on="time",
            direction="nearest",
            tolerance=pd.Timedelta("5min"),
        ).set_index("time")
        print(
            f"dust matched: {df_joined['particle_total_count'].notna().sum()} exposures"
        )

    print(f"columns after turbulence + dust join: {len(df_joined.columns)}")

## §8 — Core join: per-exposure mirror motion & force excursion

`per_exposure_mirror_stats` (reused from the PSF notebook) resamples each high-rate topic
to 1 s, assigns samples to their exposure window `[obs_start, obs_start+exp_time]`, and
reduces to per-exposure **mean** (bulk/static position, anchor c) and **std**
(in-exposure dynamic motion). From the hardpoint forces we derive **max |force|** and
**force excursion** (anchor a).

We *also* retain the raw high-rate samples sliced per exposure (`raw_slices`) so §13b can
compute Welch PSDs — the mean/std reduction alone discards the spectral content needed to
find wind-induced resonances.

In [ ]:
if not _from_cache:

    def per_exposure_mirror_stats(
        df_efd, df_exp_src, prefix, settle_skip_s=SETTLE_SKIP_S
    ):
        """Per-exposure mean (quasi-static) + std (dynamic) at NATIVE sample rate.

        Window = [obs_start + settle_skip_s, obs_start + exp_time]: the leading skip
        drops the slew-settle transient; no pre-resample so std/excursion keep the true
        ~50 Hz dynamic content (gust buffeting / oscillations) instead of 1 s averages.
        """
        if df_efd is None or df_efd.empty:
            return pd.DataFrame(index=df_exp_src.index)
        efd = df_efd.copy()
        if efd.index.tzinfo is None:
            efd.index = efd.index.tz_localize("UTC")
        cols = list(efd.columns)
        exp_win = (
            pd.DataFrame(
                {
                    "exp_start": df_exp_src.index
                    + pd.to_timedelta(settle_skip_s, unit="s"),
                    "exp_end": df_exp_src.index
                    + pd.to_timedelta(df_exp_src["exp_time"].clip(lower=1), unit="s"),
                    "exp_key": df_exp_src.index,
                }
            )
            .sort_values("exp_start")
            .reset_index(drop=True)
        )
        exp_win = exp_win[exp_win["exp_start"] < exp_win["exp_end"]]
        efd_r = efd.reset_index().rename(columns={efd.index.name or "index": "t"})
        m = pd.merge_asof(
            efd_r.sort_values("t"),
            exp_win[["exp_start", "exp_end", "exp_key"]].rename(
                columns={"exp_start": "t"}
            ),
            on="t",
            direction="backward",
            tolerance=pd.Timedelta("600s"),
        )
        m = m[m["t"] <= m["exp_end"]].dropna(subset=["exp_end"])
        if m.empty:
            return pd.DataFrame(index=df_exp_src.index)
        grp = m.groupby("exp_key")
        mean_df = grp[cols].mean().rename(columns=lambda c: f"{prefix}_{c}_mean")
        std_df = grp[cols].std().rename(columns=lambda c: f"{prefix}_{c}_std")
        stats = pd.concat([mean_df, std_df], axis=1)
        stats.index.name = df_exp_src.index.name
        return stats.reindex(df_exp_src.index)

    for key in mirror_topics:
        stats = per_exposure_mirror_stats(mirror_data.get(key), df_joined, prefix=key)
        n = stats.notna().any(axis=1).sum()
        print(f"[{key}] exposure-matched: {n} / {len(df_joined)}")
        df_joined = df_joined.join(stats, how="left")

In [ ]:
if not _from_cache:
    # ── PRIMARY force anchor: M1M3 balance MOMENT (loop disturbance response) ─
    if "m1m3bal_mx_mean" in df_joined and "m1m3bal_my_mean" in df_joined:
        df_joined["bal_moment_mag"] = np.hypot(
            df_joined["m1m3bal_mx_mean"], df_joined["m1m3bal_my_mean"]
        )
    if "m1m3bal_mx_std" in df_joined and "m1m3bal_my_std" in df_joined:
        df_joined["bal_moment_dyn_rms"] = np.hypot(
            df_joined["m1m3bal_mx_std"], df_joined["m1m3bal_my_std"]
        )

    # ── Diagnostic only: hardpoint force excursion (loop error residual) ─────
    _hp_mean = [f"m1m3hp_measuredForce{i}_mean" for i in range(6)]
    _hp_std = [f"m1m3hp_measuredForce{i}_std" for i in range(6)]
    _hp_mean = [c for c in _hp_mean if c in df_joined]
    _hp_std = [c for c in _hp_std if c in df_joined]
    if _hp_mean:
        hp_abs = df_joined[_hp_mean].abs()
        df_joined["hp_force_max_abs"] = hp_abs.max(axis=1)  # peak static load
        df_joined["hp_force_excursion"] = df_joined[_hp_mean].max(axis=1) - df_joined[
            _hp_mean
        ].min(axis=1)
    if _hp_std:
        df_joined["hp_force_dyn_rms"] = np.sqrt((df_joined[_hp_std] ** 2).mean(axis=1))

    # ── Anchor c helpers: M1M3 bulk piston & tip/tilt magnitude ──────────────
    if "m1m3ims_zPosition_mean" in df_joined:
        df_joined["m1m3_piston_um"] = df_joined["m1m3ims_zPosition_mean"]
    _tt = [
        c
        for c in ("m1m3ims_xRotation_mean", "m1m3ims_yRotation_mean")
        if c in df_joined
    ]
    if len(_tt) == 2:
        df_joined["m1m3_tilt_asec"] = np.hypot(df_joined[_tt[0]], df_joined[_tt[1]])
    # dynamic IMS motion magnitude (complements ConsDB mount jitter)
    _ims_std = [
        c
        for c in df_joined.columns
        if c.startswith("m1m3ims_") and c.endswith("Position_std")
    ]
    if _ims_std:
        df_joined["m1m3_ims_dyn_rms_um"] = np.sqrt(
            (df_joined[_ims_std] ** 2).mean(axis=1)
        )
    print("derived force/motion columns added")

In [ ]:
if not _from_cache:
    # ── Retain raw high-rate samples per exposure for §13b PSDs ──────────────
    # Keyed by exposure obs_start; only for exposures long enough for a PSD and
    # only when high-rate data exists. Stored as a dict of small DataFrames.
    raw_slices = {"m1m3ims": {}, "m1m3hp": {}}
    PSD_MIN_EXPTIME = 15.0  # s — need enough samples for low-freq resolution
    _cand = df_joined[
        (df_joined["exp_time"] >= PSD_MIN_EXPTIME) & df_joined["shutter_open"]
    ].index
    for key in ("m1m3ims", "m1m3hp"):
        raw = mirror_data.get(key)
        if raw is None or raw.empty:
            continue
        r = raw.copy()
        if r.index.tzinfo is None:
            r.index = r.index.tz_localize("UTC")
        for t0 in _cand:
            t1 = t0 + pd.to_timedelta(df_joined.loc[t0, "exp_time"], unit="s")
            seg = r.loc[t0:t1]
            if len(seg) >= 32:
                raw_slices[key][t0] = seg
    print(
        "retained raw slices — IMS:",
        len(raw_slices["m1m3ims"]),
        " HP:",
        len(raw_slices["m1m3hp"]),
    )

## §9 — Assemble analysis frame + save cache

Restrict to the analysis population (shutter-open science that can see sky) and persist.
All heavy fetch/compute above is behind `if not _from_cache:`; **every analysis section
below reads `df_joined` only**, so re-runs and the eventual full run are cheap. The
per-exposure spectral products (§13b) are cached separately once computed.

In [ ]:
if not _from_cache:
    # Guard the analysis to genuine open-dome science observing:
    #   science image type, both shutter leaves open, can see sky, not vignetted.
    # This excludes dome-closed engineering/calibration, daytime tests, and
    # shutter-transition or dome/mirror-cover-obscured frames.
    _not_vignetted = (
        ~df_joined["vignette"].astype(str).str.upper().isin(["FULLY", "PARTIALLY"])
        if "vignette" in df_joined
        else True
    )
    df_joined["analysis_ok"] = (
        (df_joined["img_type"] == "science")
        & df_joined["shutter_open"].fillna(False).astype(bool)
        & df_joined["can_see_sky"].fillna(True).astype(bool)
        & _not_vignetted
    )
    print(f"analysis population: {df_joined['analysis_ok'].sum()} / {len(df_joined)}")
    # parquet can't store interval/categorical bin columns — stringify a copy for saving
    _save = df_joined.copy()
    for c in _save.columns:
        if (
            isinstance(_save[c].dtype, pd.CategoricalDtype)
            or _save[c].dtype == object
            or str(_save[c].dtype).startswith("interval")
        ):
            try:
                _save[c] = _save[c].astype(str)
            except Exception:
                _save = _save.drop(columns=[c])
    try:
        _save.to_parquet(CACHE_FILE)
        print(f"✓ saved {len(_save):,} rows → {CACHE_FILE}")
    except Exception as e:
        print(f"⚠ cache save skipped ({str(e)[:80]}) — continuing in-memory")
else:
    print("loaded from cache; skipping save")

# Analysis frame used by all sections below (rebuild bin columns if loaded from cache)
if "analysis_ok" in df_joined.columns:
    _ok = df_joined["analysis_ok"]
    _ok = _ok if _ok.dtype == bool else _ok.astype(str).isin(["True", "true", "1"])
    df = df_joined[_ok].copy()
else:
    df = df_joined.copy()

# ensure derived categorical/interval columns exist (needed after cache reload)
if "wind_speed_bin" not in df or df["wind_speed_bin"].dtype == object:
    df["wind_speed_bin"] = pd.cut(
        df["efd_wind_speed"], bins=wind_speed_bins, labels=wind_speed_labels
    )
if "rel_wind_bin" not in df or df["rel_wind_bin"].dtype == object:
    df["rel_wind_bin"] = pd.cut(df["relative_wind"], bins=rel_wind_bins)
for _c in ("into_wind", "away_wind"):
    if _c in df and df[_c].dtype != bool:
        df[_c] = df[_c].astype(str).isin(["True", "true", "1"])
print(f"df (analysis frame): {len(df)} rows, {df.columns.size} columns")

### M1M3 VMS (accelerometer) join

The IMS in-exposure dynamic RMS sits at the sensor floor (~1e-7 native, no wind signal),
so the **mirror dynamic-motion / oscillation anchor is re-based on the M1M3 Vibration
Monitoring System** (`lsst.sal.MTVMS.data`, salIndex=1 — 3 accelerometers × 3 axes at
~240 Hz per sensor). VMS metrics are built by `build_vms.py` over a wind-stratified sample
(~30 exposures/night, settled part only) and joined here by exposure time:
`vms_accel_rms` (broadband g), band-limited RMS (0–1/1–5/5–20/20–100 Hz), and the dominant
peak frequency `vms_peak_hz`. Exposures outside the VMS sample carry NaN.

In [ ]:
# Join M1M3 VMS per-exposure metrics (wind-stratified sample) onto df, by time index.
_vms_path = CACHE_DIR / f"wind_loading_vms_{day_obs_start}_{day_obs_end}.parquet"
HAVE_VMS = _vms_path.exists()
if HAVE_VMS:
    _vms = pd.read_parquet(_vms_path)
    if not isinstance(_vms.index, pd.DatetimeIndex):
        _vms.index = pd.to_datetime(_vms.index, utc=True)
    _vcols = [c for c in _vms.columns if c.startswith("vms_")]
    df = df.join(_vms[_vcols], how="left")
    n = df["vms_accel_rms"].notna().sum() if "vms_accel_rms" in df else 0
    print(f"VMS-joined exposures: {n} / {len(df)}  (wind-stratified sample)")
    print(f"  VMS columns: {_vcols}")
else:
    print("No VMS cache found — run build_vms.py. §12/§13b will fall back to IMS.")

---
# Layer A — Measured telemetry evidence

Everything below reads the analysis frame `df` (one row per shutter-open science
exposure). We first define shared helpers for the two-sided (into-wind vs away-from-wind)
binned curves used throughout.

In [ ]:
# ── Shared binning / plotting helpers ────────────────────────────────────────
def binned_stats(x, y, bins):
    """Return bin centres, mean, std, sem, count for y binned by x."""
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    ok = np.isfinite(x) & np.isfinite(y)
    x, y = x[ok], y[ok]
    idx = np.digitize(x, bins) - 1
    cen, mean, std, sem, cnt = [], [], [], [], []
    for b in range(len(bins) - 1):
        m = idx == b
        n = m.sum()
        cen.append(0.5 * (bins[b] + bins[b + 1]))
        if n == 0:
            mean.append(np.nan)
            std.append(np.nan)
            sem.append(np.nan)
            cnt.append(0)
        else:
            mean.append(np.nanmean(y[m]))
            s = np.nanstd(y[m])
            std.append(s)
            sem.append(s / np.sqrt(n))
            cnt.append(int(n))
    return (np.array(cen), np.array(mean), np.array(std), np.array(sem), np.array(cnt))


def two_sided_curve(
    ax, df_in, metric, bins=None, ylabel=None, title=None, min_count=3, logy=False
):
    """Plot metric vs wind speed, into-wind (solid) vs away-wind (dashed)."""
    bins = speed_fit_bins if bins is None else bins
    out = {}
    for lab, mask, color, ls in [
        ("into wind (|Δ|<%d°)" % INTO_WIND_MAX, df_in["into_wind"], "firebrick", "-"),
        ("away wind (|Δ|>%d°)" % AWAY_WIND_MIN, df_in["away_wind"], "steelblue", "--"),
    ]:
        sub = df_in[mask.fillna(False)]
        c, mean, std, sem, cnt = binned_stats(sub["efd_wind_speed"], sub[metric], bins)
        keep = cnt >= min_count
        ax.errorbar(
            c[keep],
            mean[keep],
            yerr=sem[keep],
            fmt="o" + ls,
            color=color,
            capsize=3,
            label=f"{lab}  (n={int(cnt.sum())})",
            lw=1.6,
            ms=4,
        )
        out[lab] = (c, mean, std, sem, cnt)
    ax.set_xlabel("outside wind speed  [m/s]")
    if ylabel:
        ax.set_ylabel(ylabel)
    if title:
        ax.set_title(title)
    if logy:
        ax.set_yscale("log")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
    return out


def polar_by_relwind(ax, df_in, metric, speed_min=6.0, agg="mean"):
    """Polar plot of metric per 15° relative-wind bin (wind ≥ speed_min)."""
    sub = df_in[df_in["efd_wind_speed"] >= speed_min]
    c, mean, std, sem, cnt = binned_stats(
        sub["relative_wind"], sub[metric], rel_wind_bins
    )
    theta = np.deg2rad(c)
    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)
    good = cnt >= 3
    vals = np.nan_to_num(mean[good])
    ax.bar(theta[good], vals, width=np.deg2rad(14), color="teal", alpha=0.7)
    # Pin the radial axis to [0, max] so bars read from the centre outward;
    # otherwise matplotlib auto-scales to a tiny window centred on ~0 and the
    # bars vanish behind negative rings.
    if vals.size and np.nanmax(vals) > 0:
        ax.set_rmin(0)
        ax.set_rmax(np.nanmax(vals) * 1.15)
    ax.set_title(f"{metric}\n(wind ≥ {speed_min} m/s; 0°=into wind)", fontsize=9)


print("helpers ready")

## §10 — Data overview & population sanity

Before any inference: what does the sample actually cover? The exposure-count heatmap over
(wind speed × relative wind) exposes where statistics will be thin — critical for honest
limit confidence intervals in §20. We also cross-check the EFD-derived wind against the
ConsDB `wind_speed`/`wind_dir` summary.

**Population guard.** The analysis frame `df` is restricted to genuine open-dome science
observing — `img_type='science'`, **both dome shutter leaves >95 % open** at the exposure
midpoint, `can_see_sky`, and not vignetted (§9 `analysis_ok`). Dome-closed engineering,
calibration, daytime tests, and shutter-transition frames are excluded, so wind loading is
never contaminated by closed-dome or slewing periods.

**Selection-bias caveat.** Because these are *only* frames the observatory chose to take,
operations avoid high wind and tend to point away from it when gusty — so the high-wind /
into-wind corner is under-sampled by construction. The diagnostic below quantifies this;
it bounds how far the measured curves can carry the operational limit before the §17–18
aerodynamic model must take over the extrapolation.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# (1) wind speed time series with exposures overlaid
ax = axes[0, 0]
ax.plot(df.index, df["efd_wind_speed"], ".", ms=2, alpha=0.4, color="gray")
ax.set_ylabel("EFD wind speed [m/s]")
ax.set_title("Wind speed over exposures")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
ax.grid(alpha=0.3)

# (2) wind speed histogram
ax = axes[0, 1]
ax.hist(df["efd_wind_speed"].dropna(), bins=40, color="steelblue", alpha=0.8)
ax.set_xlabel("EFD wind speed [m/s]")
ax.set_ylabel("exposures")
ax.set_title(
    f"median={df['efd_wind_speed'].median():.1f}  "
    f"95th pct={df['efd_wind_speed'].quantile(0.95):.1f} m/s"
)
ax.grid(alpha=0.3)

# (3) exposure count heatmap: speed bin × rel-wind bin
ax = axes[1, 0]
sp_edges = np.array([0, 3, 6, 9, 12, 20])
rw_edges = np.arange(-180, 181, 30)
H, _, _ = np.histogram2d(
    df["efd_wind_speed"].fillna(-1),
    df["relative_wind"].fillna(-999),
    bins=[sp_edges, rw_edges],
)
im = ax.imshow(
    H,
    aspect="auto",
    origin="lower",
    cmap="viridis",
    extent=[rw_edges[0], rw_edges[-1], 0, len(sp_edges) - 1],
)
ax.set_xlabel("relative wind [deg] (0=into)")
ax.set_ylabel("speed bin")
ax.set_yticks(np.arange(len(sp_edges) - 1) + 0.5)
ax.set_yticklabels([f"{sp_edges[i]}-{sp_edges[i+1]}" for i in range(len(sp_edges) - 1)])
ax.set_title("exposure count")
plt.colorbar(im, ax=ax, fraction=0.046)

# (4) ConsDB vs EFD wind cross-check
ax = axes[1, 1]
m = df["wind_speed"].notna() & df["efd_wind_speed"].notna()
ax.scatter(df.loc[m, "wind_speed"], df.loc[m, "efd_wind_speed"], s=6, alpha=0.3)
lim = [0, np.nanmax([df["wind_speed"].max(), df["efd_wind_speed"].max()])]
ax.plot(lim, lim, "k--", lw=1)
if m.sum() > 2:
    r = np.corrcoef(df.loc[m, "wind_speed"], df.loc[m, "efd_wind_speed"])[0, 1]
    ax.set_title(f"ConsDB vs EFD wind  (r={r:.2f}, n={m.sum()})")
ax.set_xlabel("ConsDB wind_speed [m/s]")
ax.set_ylabel("EFD wind_speed [m/s]")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(
    f"into-wind exposures: {df['into_wind'].sum()}   away-wind: {df['away_wind'].sum()}"
)

In [ ]:
# ── Observing selection-bias diagnostic ─────────────────────────────────────
# Do operators avoid high wind, and point away from it when gusty? If so, the
# into-wind / high-wind regime is self-censored out of the science sample.
_m = df["efd_wind_speed"].notna() & df["abs_relative_wind"].notna()
_rho, _p = spstats.spearmanr(
    df.loc[_m, "efd_wind_speed"], df.loc[_m, "abs_relative_wind"]
)
ws = df["efd_wind_speed"]
print("Wind-speed coverage during science observing:")
print(
    f"  median={ws.median():.1f}  90th={ws.quantile(.9):.1f}  "
    f"95th={ws.quantile(.95):.1f}  99th={ws.quantile(.99):.1f}  max={ws.max():.1f} m/s"
)
print(
    f"  fraction >8 m/s: {(ws>8).mean()*100:.1f}%   >12 m/s: {(ws>12).mean()*100:.1f}%"
)
print(
    f"\nAvoidance test  Spearman(wind_speed, |relative_wind|) = {_rho:+.2f} (p={_p:.1e})"
)
print("  (+ → point more AWAY from wind as it rises = active avoidance)")
print("\nMax wind actually reached, into vs away:")
for _lab, _mask in [
    ("into (|Δ|<45°)", df["into_wind"].fillna(False)),
    ("away (|Δ|>135°)", df["away_wind"].fillna(False)),
]:
    _s = df.loc[_mask, "efd_wind_speed"]
    if _s.notna().any():
        print(
            f"  {_lab:16s} n={int(_mask.sum()):5d}  median={_s.median():.1f}  "
            f"95th={_s.quantile(.95):.1f}  max={_s.max():.1f} m/s"
        )

# visualize: |relative_wind| vs wind speed (are high-wind frames pushed to |Δ|→180?)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hexbin(
    df.loc[_m, "efd_wind_speed"],
    df.loc[_m, "abs_relative_wind"],
    gridsize=25,
    cmap="magma",
    mincnt=1,
)
ax[0].axhline(
    INTO_WIND_MAX, color="firebrick", ls="--", lw=1, label=f"into (<{INTO_WIND_MAX:g}°)"
)
ax[0].axhline(
    AWAY_WIND_MIN, color="steelblue", ls="--", lw=1, label=f"away (>{AWAY_WIND_MIN:g}°)"
)
ax[0].set_xlabel("wind speed [m/s]")
ax[0].set_ylabel("|relative wind| [deg]")
ax[0].set_title(f"pointing vs wind (Spearman={_rho:+.2f})")
ax[0].legend(fontsize=8)
# max wind reached per relative-wind bin — shows the into-wind coverage ceiling
c, mx = [], []
for b in range(len(rel_wind_bins) - 1):
    seg = df[
        (df["relative_wind"] >= rel_wind_bins[b])
        & (df["relative_wind"] < rel_wind_bins[b + 1])
    ]
    c.append(0.5 * (rel_wind_bins[b] + rel_wind_bins[b + 1]))
    mx.append(seg["efd_wind_speed"].quantile(0.95) if len(seg) > 5 else np.nan)
ax[1].plot(c, mx, "o-", color="teal")
ax[1].set_xlabel("relative wind [deg] (0=into)")
ax[1].set_ylabel("95th-pctile wind [m/s]")
ax[1].set_title("wind-speed ceiling vs pointing")
ax[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

### §10b — Does the sample reach the design point? *(gating result)*

This is the **gating fact for the entire notebook**: the verdict's form depends on whether
the data reach 20 m/s sustained / 25 m/s gust. We tabulate exposure and *night* counts above
a ladder of thresholds, separately for sustained (`efd_wind_speed`) and 1-min gust
(`efd_wind_speed_max`).

We also measure the **observed gust factor** `efd_wind_speed_max / efd_wind_speed`. The spec
pairs 20 m/s sustained with 25 m/s gusts, i.e. an implied factor of **1.25**. If the observed
distribution brackets 1.25, the spec's own sustained/gust pairing is internally consistent
with the site's statistics — an independent check that costs nothing and justifies the
gust-by-scaling treatment used in §20.

Night counts matter more than exposure counts: with night-level confounding (§16), the
**effective sample size for a wind effect is the number of distinct windy nights**, not the
number of exposures. Extending the window from 73 to 116 nights more than doubled the windy
nights (4 → 9 above 14 m/s sustained), which is where the added statistical power comes from.

> **Two wind columns, one of them a trap.** ConsDB `wind_speed` is a per-exposure summary that
> is held/stale in the high-wind tail (it reaches 20.6 m/s on 2026-05-26 by repeating a single
> value), while `efd_wind_speed` is a fresh 1-min EFD resample topping out at 18.9 m/s on the
> same night. The table below reports the EFD metric; treating the ConsDB value as the truth
> would falsely claim the 20 m/s spec point had been reached.


In [ ]:
# ── §10b — spec-reach coverage: how close does the sample get to 20/25 m/s? ──
SPEC_SUSTAINED = 20.0  # m/s — degraded-ops sustained design wind
SPEC_GUST = 25.0  # m/s — degraded-ops gust design wind
SPEC_GUST_FACTOR = SPEC_GUST / SPEC_SUSTAINED  # 1.25, used for gust scaling in §20

_V = pd.to_numeric(df["efd_wind_speed"], errors="coerce")
_G = (
    pd.to_numeric(df["efd_wind_speed_max"], errors="coerce")
    if "efd_wind_speed_max" in df
    else pd.Series(np.nan, index=df.index)
)

print("=" * 78)
print("SPEC-REACH COVERAGE — open-dome science exposures")
print("=" * 78)
print(f"  exposures: {len(df):,}   nights: {df['day_obs'].nunique()}")
print(
    f"  sustained wind: median {_V.median():.2f}  p99 {_V.quantile(.99):.2f}  MAX {_V.max():.2f} m/s"
)
print(
    f"  1-min gust:     median {_G.median():.2f}  p99 {_G.quantile(.99):.2f}  MAX {_G.max():.2f} m/s"
)
print(
    f"\n  {'thresh':>7s} | {'sustained exp':>13s} {'nights':>7s} | {'gust exp':>9s} {'nights':>7s}"
)
print("  " + "-" * 60)
_cov = []
for _t in [8, 10, 12, 14, 15, 16, 17, 18, 20, 25]:
    _ms, _mg = _V > _t, _G > _t
    _ns = df.loc[_ms.fillna(False), "day_obs"].nunique()
    _ng = df.loc[_mg.fillna(False), "day_obs"].nunique()
    _star = "  ← SPEC" if _t in (20, 25) else ""
    print(
        f"  {_t:6.0f}s | {int(_ms.sum()):13d} {_ns:7d} | {int(_mg.sum()):9d} {_ng:7d}{_star}"
    )
    _cov.append(
        {
            "thresh": _t,
            "n_sustained": int(_ms.sum()),
            "nights_sustained": _ns,
            "n_gust": int(_mg.sum()),
            "nights_gust": _ng,
        }
    )
cov_df = pd.DataFrame(_cov)

# Observed gust factor vs the spec's implied 25/20 = 1.25
_gf = (_G / _V).where(_V > 3)  # ratio is noise-dominated at very low speed
_gf = _gf[np.isfinite(_gf)]
SPEC_REACHED = bool((_V > SPEC_SUSTAINED).sum() > 0)
GUST_REACHED = bool((_G > SPEC_GUST).sum() > 0)
V_MAX_OBS, G_MAX_OBS = float(_V.max()), float(_G.max())
print(
    f"\n  observed gust factor (V>3): median {_gf.median():.3f}  p90 {_gf.quantile(.90):.3f}  p99 {_gf.quantile(.99):.3f}"
)
print(
    f"  spec implied gust factor: {SPEC_GUST_FACTOR:.2f}  → "
    f"{'BRACKETED by observations (spec pairing self-consistent)' if _gf.median() <= SPEC_GUST_FACTOR <= _gf.quantile(.99) else 'NOT bracketed — revisit gust scaling'}"
)
print(
    f"\n  ⇒ sustained {SPEC_SUSTAINED:.0f} m/s reached? {'YES' if SPEC_REACHED else 'NO'}"
    f"   gust {SPEC_GUST:.0f} m/s reached? {'YES' if GUST_REACHED else 'NO'}"
)
print(
    f"  ⇒ every statement about the design point is an EXTRAPOLATION from V_max="
    f"{V_MAX_OBS:.1f} m/s (gust {G_MAX_OBS:.1f})."
)
# ── Guard: ConsDB `wind_speed` is HELD/STALE in the high-wind tail ───────────
# It can exceed the EFD metric enough to look like the spec point was reached.
# Confirm which column drives the analysis and quantify the disagreement.
if "wind_speed" in df:
    _cw = pd.to_numeric(df["wind_speed"], errors="coerce")
    _n_cdb_over = int((_cw > SPEC_SUSTAINED).sum())
    _n_efd_over = int((_V > SPEC_SUSTAINED).sum())
    _both = pd.DataFrame({"c": _cw, "e": _V}).dropna()
    print("\n  wind-column cross-check (ConsDB summary vs EFD 1-min resample):")
    print(
        f"    ConsDB wind_speed : max {_cw.max():.2f} m/s, {_n_cdb_over} exposures > {SPEC_SUSTAINED:.0f}"
    )
    print(
        f"    EFD efd_wind_speed: max {_V.max():.2f} m/s, {_n_efd_over} exposures > {SPEC_SUSTAINED:.0f}   <-- USED HERE"
    )
    print(
        f"    agreement r={_both['c'].corr(_both['e']):.3f}, median ratio {(_both['c']/_both['e']).median():.3f}"
    )
    if _n_cdb_over > _n_efd_over:
        _wn = sorted(df.loc[(_cw > SPEC_SUSTAINED).fillna(False), "day_obs"].unique())
        print(
            f"    WARNING: ConsDB alone would claim the {SPEC_SUSTAINED:.0f} m/s spec point was"
        )
        print(
            f"      REACHED on night(s) {_wn} — but it is a held summary there (few distinct"
        )
        print(
            f"      values repeating), so that claim is spurious. Use the EFD metric."
        )

print(
    f"  ⇒ V² extrapolation inflates slope uncertainty by ({SPEC_SUSTAINED/V_MAX_OBS:.2f})² = "
    f"{(SPEC_SUSTAINED/V_MAX_OBS)**2:.2f}x at 20 m/s, "
    f"{(SPEC_GUST/V_MAX_OBS)**2:.2f}x at 25 m/s."
)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
ax = axes[0]
ax.hist(_V.dropna(), bins=np.arange(0, 27, 0.5), color="steelblue", label="sustained")
ax.hist(
    _G.dropna(),
    bins=np.arange(0, 27, 0.5),
    histtype="step",
    color="firebrick",
    lw=1.5,
    label="1-min gust",
)
for _x, _c, _l in [
    (SPEC_SUSTAINED, "k", "20 m/s spec"),
    (SPEC_GUST, "gray", "25 m/s gust spec"),
]:
    ax.axvline(_x, color=_c, ls="--", lw=1.6, label=_l)
ax.axvspan(V_MAX_OBS, 27, color="red", alpha=0.07)
ax.text(
    0.97,
    0.55,
    "NO DATA",
    color="firebrick",
    fontsize=9,
    rotation=90,
    ha="right",
    va="center",
    transform=ax.transAxes,
)
ax.set_yscale("log")
ax.set_xlabel("wind speed [m/s]")
ax.set_ylabel("exposures")
ax.set_title("Sample vs design point")
ax.legend(fontsize=7)
ax.grid(alpha=0.3)

ax = axes[1]
ax.semilogy(
    cov_df["thresh"],
    cov_df["n_sustained"].clip(lower=0.5),
    "o-",
    color="steelblue",
    label="sustained",
)
ax.semilogy(
    cov_df["thresh"],
    cov_df["n_gust"].clip(lower=0.5),
    "s--",
    color="firebrick",
    label="gust",
)
ax.axvline(SPEC_SUSTAINED, color="k", ls="--", lw=1.5)
ax.axvline(SPEC_GUST, color="gray", ls="--", lw=1.5)
ax.axhline(1, color="k", lw=0.6)
ax.set_xlabel("threshold [m/s]")
ax.set_ylabel("exposures above (0.5 = none)")
ax.set_title("Exposure count above threshold")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

ax = axes[2]
ax.hist(_gf, bins=np.linspace(1, 2.2, 50), color="mediumseagreen")
ax.axvline(
    SPEC_GUST_FACTOR,
    color="k",
    ls="--",
    lw=1.8,
    label=f"spec 25/20 = {SPEC_GUST_FACTOR:.2f}",
)
ax.axvline(
    _gf.median(),
    color="firebrick",
    ls="-",
    lw=1.4,
    label=f"observed median {_gf.median():.2f}",
)
ax.axvline(
    _gf.quantile(0.90),
    color="goldenrod",
    ls=":",
    lw=1.4,
    label=f"observed p90 {_gf.quantile(.90):.2f}",
)
ax.set_xlabel("gust factor  V_max / V_mean")
ax.set_ylabel("exposures")
ax.set_title("Observed vs spec gust factor")
ax.legend(fontsize=7)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## §11 — Measured aerodynamic loading: force & moment vs wind

**Primary force anchor: the M1M3 balance MOMENT** (`appliedBalanceForces` mx,my). The
force-balance loop actively drives the hardpoint loads toward zero, carrying the mirror on
the static support; what the loop *applies* is its response to disturbances (wind, thermal),
so the balance moment is the physical wind-load signal. This also gives the moment about the
mirror axes **directly** — no lever-arm assumption needed. (The raw hardpoint forces are the
loop *error residual* — ≈0 in steady state and dominated by slew-settle transients — so they
are retained only as a diagnostic, not the anchor.)

The aerodynamic prediction is that the wind moment scales with dynamic pressure `q ∝ V²`, so
we plot the wind moment against `V²` and look for linearity. The moment responds to the *net*
disturbance (elevation-dependent gravity residual + thermal + wind), so we isolate the wind
term by subtracting a per-elevation calm-wind baseline (median at V < 2 m/s in the same
elevation stratum); the residual is `wind_moment`.

Two data-quality controls (both verified to matter): the per-exposure statistics **skip the
first `SETTLE_SKIP_S` = 5 s** of each exposure (slew-settle transient — dynamic RMS ~halves
when excluded) and are computed at the **native ~50 Hz** rate (no 1 s pre-averaging, so the
dynamic content is preserved).

In [ ]:
# PRIMARY force anchor: M1M3 balance MOMENT (the force-balance loop's disturbance
# response). Isolate the wind-driven part via a per-elevation calm-wind baseline
# (gravity/elevation term removed), leaving the wind moment.
FORCE_METRIC = "bal_moment_mag" if "bal_moment_mag" in df else "hp_force_excursion"
if FORCE_METRIC in df:
    df["_el_bin"] = pd.cut(
        df["mount_el_actual"].fillna(df["altitude"]), bins=np.arange(0, 91, 15)
    )
    low_wind = df[df["efd_wind_speed"] < 2.0]
    base = low_wind.groupby("_el_bin", observed=True)[FORCE_METRIC].median()
    df["wind_moment"] = (df[FORCE_METRIC] - df["_el_bin"].map(base)).clip(lower=0)
    HAVE_FORCE = df["wind_moment"].notna().sum() > 10
    _unit = "N·m" if FORCE_METRIC == "bal_moment_mag" else "N"
else:
    HAVE_FORCE = False
    print("⚠ no balance-moment / hardpoint force data — anchor (a) unavailable")

if HAVE_FORCE:
    print(
        f"force anchor = {FORCE_METRIC} (wind moment, per-elevation-baseline-subtracted, {_unit})"
    )
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    # (1) wind moment vs V^2 hexbin — aero prediction is linear in V²
    ax = axes[0]
    m = df["efd_wind_speed"].notna() & df["wind_moment"].notna()
    ax.hexbin(
        df.loc[m, "efd_wind_speed"] ** 2,
        df.loc[m, "wind_moment"],
        gridsize=30,
        cmap="inferno",
        mincnt=1,
    )
    ax.set_xlabel("V²  [m²/s²]")
    ax.set_ylabel(f"wind moment [{_unit}]")
    ax.set_title(f"{FORCE_METRIC} wind term vs V² (density)")
    # (2) two-sided curve
    two_sided_curve(
        axes[1],
        df,
        "wind_moment",
        ylabel=f"wind moment [{_unit}]",
        title="M1M3 wind moment vs wind speed",
    )
    # (3) polar
    fig.delaxes(axes[2])
    axes[2] = fig.add_subplot(1, 3, 3, projection="polar")
    polar_by_relwind(axes[2], df, "wind_moment", speed_min=6.0)
    plt.tight_layout()
    plt.show()
    # diagnostic: HP excursion (loop error residual) for comparison — should be small/noisy
    if "hp_force_excursion" in df:
        _r = df[["wind_moment", "hp_force_excursion"]].corr().iloc[0, 1]
        print(
            f"(diagnostic) corr(wind_moment, HP-excursion residual) = {_r:.2f} "
            "— HP forces are the loop error, not the load"
        )

## §12 — Mirror bulk motion vs wind  *(anchor c)*

How far the mirror physically moves under wind: M1M3 piston (`zPosition`) and tip/tilt
(`x/yRotation`), plus M2. We also cross-check IMS motion against hardpoint force excursion
— both should track the same aerodynamic load.

In [ ]:
# Bulk static position from IMS mean; DYNAMIC motion from M1M3 VMS accelerometers
# (the IMS in-exposure std is at the sensor floor, so VMS is the dynamic anchor c).
HAVE_IMS = "m1m3_piston_um" in df and df["m1m3_piston_um"].notna().any()
HAVE_VMS_MOTION = "vms_accel_rms" in df and df["vms_accel_rms"].notna().sum() > 10

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
# (1) IMS bulk static position (piston) vs wind — quasi-static deflection
if HAVE_IMS:
    two_sided_curve(
        axes[0],
        df,
        "m1m3_piston_um",
        ylabel="M1M3 piston (IMS mean) [native]",
        title="M1M3 bulk position vs wind",
    )
else:
    axes[0].axis("off")
# (2) VMS broadband accel RMS vs wind — DYNAMIC anchor (c)
if HAVE_VMS_MOTION:
    two_sided_curve(
        axes[1],
        df,
        "vms_accel_rms",
        ylabel="VMS accel RMS [g]",
        title="M1M3 VMS dynamic accel vs wind",
    )
else:
    axes[1].axis("off")
    print("⚠ no VMS data joined — run build_vms.py; dynamic anchor (c) unavailable")
# (3) VMS dynamic accel vs measured wind moment — mechanical consistency
ax = axes[2]
if HAVE_VMS_MOTION and HAVE_FORCE:
    m = df["wind_moment"].notna() & df["vms_accel_rms"].notna()
    ax.scatter(
        df.loc[m, "wind_moment"],
        df.loc[m, "vms_accel_rms"],
        s=10,
        alpha=0.4,
        c=df.loc[m, "efd_wind_speed"],
        cmap="viridis",
    )
    ax.set_xlabel("wind moment [N·m]")
    ax.set_ylabel("VMS accel RMS [g]")
    ax.set_title("dynamic vibration vs wind moment (color=wind)")
    ax.grid(alpha=0.3)
else:
    ax.axis("off")
plt.tight_layout()
plt.show()

## §13 — Mount jitter & image degradation vs wind  *(anchor b)*

ConsDB `mount_jitter_rms` and `mount_motion_image_degradation` (total + az/el/rot), which
quantify how much the bulk telescope tracking error blurs the image. We split into/away and
decompose by axis — pointing into the wind should load the elevation/pitch axis differently
than crosswind. Elevation is a known covariate (handled in §16).

In [ ]:
jit_total = "mount_motion_image_degradation"
HAVE_JIT = jit_total in df and df[jit_total].notna().sum() > 10
if not HAVE_JIT:
    print("⚠ no mount-motion data in this window — anchor (b) unavailable")
else:
    # clip non-physical values
    for c in [jit_total, "mount_jitter_rms"]:
        if c in df:
            df.loc[(df[c] < MOUNT_JITTER_MIN) | (df[c] > MOUNT_JITTER_MAX), c] = np.nan
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    two_sided_curve(
        axes[0],
        df,
        jit_total,
        ylabel="image degradation [arcsec]",
        title="mount image degradation vs wind",
    )
    # az vs el decomposition (binned means vs speed, into-wind only)
    ax = axes[1]
    sub = df[df["into_wind"].fillna(False)]
    for axis_col, color in [
        ("mount_motion_image_degradation_az", "tab:blue"),
        ("mount_motion_image_degradation_el", "tab:red"),
        ("mount_motion_image_degradation_rot", "tab:green"),
    ]:
        if axis_col in df:
            c, mean, std, sem, cnt = binned_stats(
                sub["efd_wind_speed"], sub[axis_col], speed_fit_bins
            )
            k = cnt >= 3
            ax.plot(
                c[k], mean[k], "o-", color=color, ms=4, label=axis_col.split("_")[-1]
            )
    ax.set_xlabel("wind speed [m/s]")
    ax.set_ylabel("image deg [arcsec]")
    ax.set_title("axis decomposition (into-wind)")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    fig.delaxes(axes[2])
    axes[2] = fig.add_subplot(1, 3, 3, projection="polar")
    polar_by_relwind(axes[2], df, jit_total, speed_min=6.0)
    plt.tight_layout()
    plt.show()

## §13b — Wind-induced oscillations (M1M3 VMS dynamic / spectral response)

The IMS in-exposure dynamic RMS is at the sensor floor, so the oscillation analysis uses
the **M1M3 Vibration Monitoring System** (`lsst.sal.MTVMS.data`, salIndex=1): 3
accelerometers × 3 axes at ~240 Hz per sensor, sampled over a wind-stratified set of
exposures (§ VMS join) with per-exposure Welch PSDs computed on the settled part of each
exposure (`build_vms.py`). We show:

1. the **wind-speed-stacked median acceleration PSD** — a spectral peak whose amplitude
   grows with wind speed is a wind-driven **structural resonance**;
2. **band-limited accel RMS** (0–1 / 1–5 / 5–20 / 20–100 Hz) vs wind speed;
3. the **dominant peak frequency** distribution, low vs high wind.

Broadband buffeting grows smoothly with V²; a narrowband resonance sits at a fixed
frequency with wind-dependent amplitude.

In [ ]:
# VMS PSD products are precomputed by build_vms.py; just confirm availability here.
_vms_stk = CACHE_DIR / f"wind_loading_vmsstack_{day_obs_start}_{day_obs_end}.parquet"
HAVE_VMS_PSD = "vms_accel_rms" in df and df["vms_accel_rms"].notna().sum() > 10
HAVE_VMS_STACK = _vms_stk.exists()
print(
    f"VMS per-exposure spectral: {'available' if HAVE_VMS_PSD else 'NONE'}"
    + (f" ({int(df['vms_accel_rms'].notna().sum())} exp)" if HAVE_VMS_PSD else "")
)
print(f"VMS stacked PSD: {'available' if HAVE_VMS_STACK else 'NONE'}")

In [ ]:
if HAVE_VMS_PSD or HAVE_VMS_STACK:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    # (1) wind-stacked median acceleration PSD — resonance hunt
    ax = axes[0]
    if HAVE_VMS_STACK:
        st = pd.read_parquet(_vms_stk)
        for (lo_v, hi_v), col in zip(
            sorted(set(zip(st["wind_lo"], st["wind_hi"]))),
            ["steelblue", "goldenrod", "firebrick"],
        ):
            seg = st[(st["wind_lo"] == lo_v) & (st["wind_hi"] == hi_v)].sort_values(
                "freq_hz"
            )
            if len(seg) < 3:
                continue
            n = int(seg["n_exp"].iloc[0])
            lab = f"wind {lo_v}-{'∞' if hi_v>=90 else hi_v} m/s (n={n})"
            ax.loglog(seg["freq_hz"][1:], seg["psd_median"][1:], color=col, label=lab)
        ax.set_xlabel("frequency [Hz]")
        ax.set_ylabel("accel PSD [g²/Hz]")
        ax.set_title("M1M3 VMS wind-stacked median PSD")
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3, which="both")
    else:
        ax.axis("off")
    # (2) band-limited accel RMS vs wind speed
    ax = axes[1]
    if HAVE_VMS_PSD:
        band_cols = [c for c in df.columns if c.startswith("vms_accel_rms_")]
        cols = ["steelblue", "seagreen", "goldenrod", "firebrick"]
        for bc, col in zip(band_cols, cols):
            c, mean, std, sem, cnt = binned_stats(
                df["efd_wind_speed"], df[bc], speed_fit_bins
            )
            k = cnt >= 3
            ax.plot(
                c[k],
                mean[k],
                "o-",
                color=col,
                ms=4,
                label=bc.replace("vms_accel_rms_", "").replace("_", "-"),
            )
        ax.set_xlabel("wind speed [m/s]")
        ax.set_ylabel("band accel RMS [g]")
        ax.set_title("M1M3 VMS oscillation band-RMS vs wind")
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)
    else:
        ax.axis("off")
    # (3) dominant peak frequency, low vs high wind
    ax = axes[2]
    if HAVE_VMS_PSD and "vms_peak_hz" in df:
        lo = df[df["efd_wind_speed"] < 5]["vms_peak_hz"].dropna()
        hi = df[df["efd_wind_speed"] >= 5]["vms_peak_hz"].dropna()
        ax.hist(
            [lo, hi],
            bins=20,
            label=["wind<5", "wind≥5"],
            color=["steelblue", "firebrick"],
            alpha=0.7,
        )
        ax.set_xlabel("dominant peak freq [Hz]")
        ax.set_ylabel("exposures")
        ax.set_title("M1M3 VMS peak frequency")
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)
    else:
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("VMS spectral section skipped — run build_vms.py to populate the VMS caches.")

## §14 — Units, budget constants, and why the IQ anchors were dropped

This cell now serves a narrower purpose than in earlier revisions: it fixes the **unit
conversions and budget constants** used by the tracking anchors, and documents the removal
of the image-quality anchors.

**Why PSF FWHM and donut blur are no longer anchors.** Their apparent wind response is a
**between-night** effect that vanishes within nights (r(V², PSF) = +0.24 between nights vs
**+0.02 within** nights). Two supporting arguments in the earlier analysis were estimator
artifacts: DIMM *anti*-correlates with wind here, making it a **suppressor variable** whose
removal inflates the correlation by construction; and quadrature-subtracting the atmosphere
clipped negative residuals to zero, manufacturing a positive trend. A within-night placebo
(wind shuffled inside each night) reproduced most of the claimed signal. The corrected
treatment — where the driver is **mirror–air ΔT at ≈0.10″/K** and wind adds nothing once ΔT
is controlled — lives in
[`Dome_Seeing_Model_Construction.ipynb`](Dome_Seeing_Model_Construction.ipynb).

For the spec question this is a *narrowing*, not a loss: the within-night bound on the wind
term, |k| < 7.2×10⁻⁴ arcsec/(m/s)², caps any dome-seeing wind contribution at **< 0.29″ even
at 20 m/s**, which is carried into §22 as a side constraint rather than a binding anchor.

**Tracking-jitter budget.** Anchors (b) and (e) use the **wind-buffeting allocation 0.047″**,
not the 0.01″ total tracking-jitter requirement — the latter covers all error sources, while
this analysis isolates wind. Guider magnitude RMS is in **milli-arcsec** and divided by 1000.


In [ ]:
# ── Unit conversions + budget constants (IQ anchors intentionally NOT defined) ──
PIXSCALE = 0.2  # arcsec/pix (LSSTCam)

# Tracking-jitter budget for the bulk-motion anchors (mount_jitter_rms, guider RMS).
# This analysis isolates the WIND-driven contribution, so the anchor budget is the
# WIND-BUFFETING ALLOCATION (0.047"), NOT the total tracking-jitter design
# requirement (0.01"), which covers all error sources (servo, encoder, thermal, wind).
WIND_JITTER_ALLOCATION = 0.047  # arcsec — wind-buffeting jitter allocation
TRACK_JITTER_REQUIREMENT = 0.01  # arcsec — total design requirement (context only)
TRACK_JITTER_LIMIT = WIND_JITTER_ALLOCATION  # anchor budget = wind allocation

# Within-night 95% bound on the dome-seeing wind term, carried from
# Dome_Seeing_Model_Construction.ipynb as a SIDE CONSTRAINT (not an anchor).
DOME_SEEING_K_BOUND = 7.2e-4  # arcsec / (m/s)^2, 95% upper limit

# ConsDB numeric columns can arrive as Decimal/object — coerce to float.
for _c in [
    "guider_magnitude_rms_detrended",
    "mount_jitter_rms",
    "mount_motion_image_degradation",
]:
    if _c in df:
        df[_c] = pd.to_numeric(df[_c], errors="coerce")

# Guider magnitude RMS is reported in MILLI-arcsec (median ~5 mas, same scale as
# mount_jitter_rms in arcsec) → convert to arcsec so it shares the jitter budget.
if "guider_magnitude_rms_detrended" in df:
    df["guider_rms_asec"] = df["guider_magnitude_rms_detrended"] / 1000.0

print("Budget constants:")
print(
    f"  wind-buffeting jitter allocation : {WIND_JITTER_ALLOCATION:.3f} arcsec  (anchor budget)"
)
print(
    f"  total tracking-jitter requirement: {TRACK_JITTER_REQUIREMENT:.3f} arcsec  (context)"
)
print(
    f"  dome-seeing wind bound (side constraint): |k| < {DOME_SEEING_K_BOUND:.1e} arcsec/(m/s)^2"
)
print(
    f'    → dome-seeing wind contribution < {DOME_SEEING_K_BOUND*SPEC_SUSTAINED**2:.2f}" at '
    f'{SPEC_SUSTAINED:.0f} m/s, < {DOME_SEEING_K_BOUND*SPEC_GUST**2:.2f}" at {SPEC_GUST:.0f} m/s'
)
print(
    "\nIQ anchors (PSF FWHM, donut blur) are NOT defined as anchors: their wind response"
)
print("is a between-night artifact (r within-night = +0.02). See markdown above.")

## §14b — Guider RMS motion vs wind *(anchor e)*

`guider_magnitude_rms_detrended` integrates the *actual* image motion during the exposure,
largely free of the atmospheric-seeing confound — making it the cleanest **mechanical**
image-motion anchor, and the one IQ-adjacent metric that survives the §16 within-night test
as a usable anchor (it measures motion, not blur). We split into/away, decompose by axis
(altitude / azimuth / field-rotation), and cross-check against `mount_jitter_rms` (§13) and
the §13b oscillation band-RMS — do wind-driven resonances show up in the guider?

Budget: the **0.047″ wind-buffeting allocation** (values converted mas→arcsec).


In [ ]:
g_primary = (
    "guider_rms_asec" if "guider_rms_asec" in df else "guider_magnitude_rms_detrended"
)
HAVE_GUIDER = g_primary in df and df[g_primary].notna().sum() > 10
if not HAVE_GUIDER:
    print("⚠ no guider RMS data in this window — anchor (e) unavailable")
else:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    # (1) guider RMS (arcsec) AND mount jitter (arcsec) vs wind, with 0.01" limit
    out = two_sided_curve(
        axes[0],
        df,
        g_primary,
        ylabel="tracking RMS [arcsec]",
        title="guider RMS motion vs wind",
    )
    if "mount_jitter_rms" in df:
        for case, mask, col, ls in [
            ("into", df["into_wind"].fillna(False), "darkred", ":"),
            ("away", df["away_wind"].fillna(False), "navy", ":"),
        ]:
            sub = df[mask]
            c, mean, std, sem, cnt = binned_stats(
                sub["efd_wind_speed"], sub["mount_jitter_rms"], speed_fit_bins
            )
            k = cnt >= 3
            axes[0].plot(
                c[k], mean[k], ls, color=col, lw=1.4, label=f"mount_jitter {case}"
            )
    axes[0].axhline(
        TRACK_JITTER_LIMIT,
        color="k",
        ls="--",
        lw=1.5,
        label=f'wind-buffeting allocation {TRACK_JITTER_LIMIT}"',
    )
    axes[0].axhline(
        TRACK_JITTER_REQUIREMENT,
        color="gray",
        ls=":",
        lw=1,
        label=f'total jitter requirement {TRACK_JITTER_REQUIREMENT}" (context)',
    )
    axes[0].legend(fontsize=7)
    # (2) axis decomposition (into-wind) — native per-axis guider RMS
    ax = axes[1]
    sub = df[df["into_wind"].fillna(False)]
    for gc, color in [
        ("guider_altitude_rms_detrended", "tab:red"),
        ("guider_azimuth_rms_detrended", "tab:blue"),
        ("guider_focalplane_theta_rms_detrended", "tab:green"),
    ]:
        if gc in df:
            c, mean, std, sem, cnt = binned_stats(
                sub["efd_wind_speed"],
                pd.to_numeric(sub[gc], errors="coerce"),
                speed_fit_bins,
            )
            k = cnt >= 3
            ax.plot(c[k], mean[k], "o-", color=color, ms=4, label=gc.split("_")[1])
    ax.set_xlabel("wind speed [m/s]")
    ax.set_ylabel("guider per-axis RMS [native]")
    ax.set_title("guider axis decomposition (into-wind)")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    # (3) guider RMS vs mount jitter (both arcsec) with 1:1 and limit box
    ax = axes[2]
    if "mount_jitter_rms" in df:
        m = df[g_primary].notna() & df["mount_jitter_rms"].notna()
        ax.scatter(
            df.loc[m, "mount_jitter_rms"],
            df.loc[m, g_primary],
            s=8,
            alpha=0.3,
            c=df.loc[m, "efd_wind_speed"],
            cmap="viridis",
        )
        lim = max(
            df.loc[m, "mount_jitter_rms"].quantile(0.99),
            df.loc[m, g_primary].quantile(0.99),
            TRACK_JITTER_LIMIT * 1.5,
        )
        ax.plot([0, lim], [0, lim], "k-", lw=0.8, alpha=0.5)
        ax.axhline(TRACK_JITTER_LIMIT, color="k", ls="--", lw=1)
        ax.axvline(TRACK_JITTER_LIMIT, color="k", ls="--", lw=1)
        ax.set_xlim(0, lim)
        ax.set_ylim(0, lim)
        ax.set_xlabel("mount_jitter_rms [arcsec]")
        ax.set_ylabel("guider RMS [arcsec]")
        if m.sum() > 2:
            r = np.corrcoef(df.loc[m, "mount_jitter_rms"], df.loc[m, g_primary])[0, 1]
            ax.set_title(f"guider vs mount jitter (r={r:.2f}, both arcsec)")
        ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
    print(
        f"guider RMS: {(df[g_primary]>TRACK_JITTER_LIMIT).mean()*100:.1f}% of exposures "
        f'exceed the {TRACK_JITTER_LIMIT}" wind-buffeting allocation '
        f'({(df[g_primary]>TRACK_JITTER_REQUIREMENT).mean()*100:.0f}% exceed the {TRACK_JITTER_REQUIREMENT}" total requirement)'
    )
    if "mount_jitter_rms" in df:
        print(
            f"mount jitter: {(df['mount_jitter_rms']>TRACK_JITTER_LIMIT).mean()*100:.1f}% "
            f'exceed the {TRACK_JITTER_LIMIT}" wind allocation'
        )

## §15 — Dust ingression vs wind

Particulate influx through the open shutter/louvers. We attach `particle_total_count`
(sum of `numberConcentration0..4`, per-index median over each exposure window) to the
exposures and study it vs wind speed and relative direction — pointing *into* the wind
should ingest more through the open aperture. Louver azimuths (`../data/louver_map.csv`)
give context on which openings face the wind. This feeds a **separate contamination note**,
not the five IQ/mechanical anchors. Gracefully reports "no data" if the sensors are empty.

In [ ]:
dust_summary = {"available": False}
HAVE_DUST = (
    "particle_total_count" in df and df["particle_total_count"].notna().sum() > 10
)
if not HAVE_DUST:
    print(
        "No dust/particle ingression data available for this window — section skipped."
    )
else:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    two_sided_curve(
        axes[0],
        df,
        "particle_total_count",
        ylabel="particle count",
        title="dust vs wind speed",
        logy=True,
    )
    fig.delaxes(axes[1])
    axes[1] = fig.add_subplot(1, 3, 2, projection="polar")
    polar_by_relwind(axes[1], df, "particle_total_count", speed_min=4.0)
    ax = axes[2]
    ax.plot(
        df.index, df["particle_total_count"], ".", ms=3, color="saddlebrown", alpha=0.5
    )
    ax.set_ylabel("particle count", color="saddlebrown")
    ax.set_yscale("log")
    ax2 = ax.twinx()
    ax2.plot(df.index, df["efd_wind_speed"], ".", ms=2, color="steelblue", alpha=0.3)
    ax2.set_ylabel("wind [m/s]", color="steelblue")
    ax.set_title("dust & wind over time")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    # ── Quantitative ingression summary (carried to §22) ────────────────────
    d = df[df["particle_total_count"].notna() & df["efd_wind_speed"].notna()]
    into = d[d["into_wind"].fillna(False) & (d["efd_wind_speed"] > 4)][
        "particle_total_count"
    ]
    away = d[d["away_wind"].fillna(False) & (d["efd_wind_speed"] > 4)][
        "particle_total_count"
    ]
    med_into, med_away = into.median(), away.median()
    ratio = (
        med_into / med_away
        if med_away and np.isfinite(med_away) and med_away > 0
        else np.nan
    )
    # wind-speed dependence: Spearman correlation + low-vs-high-wind ratio
    rho, pval = spstats.spearmanr(d["efd_wind_speed"], d["particle_total_count"])
    lo = d[d["efd_wind_speed"] < 4]["particle_total_count"].median()
    hi = d[d["efd_wind_speed"] >= 8]["particle_total_count"].median()
    wind_ratio = hi / lo if lo and np.isfinite(lo) and lo > 0 else np.nan
    dust_summary = {
        "available": True,
        "n": int(len(d)),
        "median_into_gt4": med_into,
        "median_away_gt4": med_away,
        "into_away_ratio": ratio,
        "spearman_rho_vs_wind": rho,
        "spearman_p": pval,
        "median_lowwind_lt4": lo,
        "median_highwind_ge8": hi,
        "highwind_lowwind_ratio": wind_ratio,
    }
    print(f"\nIngression summary ({len(d)} exposures with dust + wind):")
    print(f"  median particle count  into-wind (>4 m/s): {med_into:.0f}")
    print(f"  median particle count  away-wind (>4 m/s): {med_away:.0f}")
    print(
        f"  into/away ratio                          : {ratio:.2f}"
        + (
            "  (into-wind ingests more — consistent with open-aperture influx)"
            if ratio == ratio and ratio > 1
            else ""
        )
    )
    print(f"  dust vs wind speed  Spearman rho={rho:+.2f} (p={pval:.1e})")
    print(f"  high-wind(≥8)/low-wind(<4) median ratio  : {wind_ratio:.2f}")

## §15b — Dome interior wind coupling: is inside wind < 50% of outside?

A standing design expectation is that the dome attenuates the wind so that **interior air
speed stays below 50% of the outside speed**. This matters for the spec verification: if the
interior flow at the 20 m/s design point stays ~10% of outside, the optics and structure see
a far gentler environment than the free-stream number suggests, and it supports the
mechanical anchors' insensitivity found in §11–§14b.

We test it with the `ESS.airTurbulence` sonic anemometers — index 110 (TMA platform) and
123–126 (top ring) — against the outside `ESS.airFlow` speed.

**Two traps, both handled:**

1. **The ratio is not scale-free.** Interior speed has a **wind-independent floor** (HVAC,
   thermal convection, dome venting), so `inside/outside` *blows up* as outside → 0. Quoting
   a raw ratio over all exposures conflates that floor with wind ingress. We therefore (i)
   restrict the ratio to a meaningful outside-speed floor, and (ii) fit
   `inside = a + b·outside`, where the **slope `b` is the true asymptotic coupling** and the
   intercept `a` isolates the wind-independent floor.
2. **Sensor saturation.** The top-ring channels saturate at a fill value; per the sibling
   notebook convention we clip at **8.6 m/s** and report how many samples that removes.

The verdict is reported both ways: the literal "always <50%" claim, and the physically
meaningful high-wind / asymptotic version.


In [ ]:
# ── §15b — inside vs outside wind speed coupling; test the <50% expectation ──
COUPLING_LIMIT = 0.50  # design expectation: inside < 50% of outside
RATIO_V_FLOOR = 8.0  # m/s — outside-speed floor for the high-wind ratio test
_inside_cols = {
    110: "TMA platform",
    123: "top ring 123",
    124: "top ring 124",
    125: "top ring 125",
    126: "top ring 126",
}
_clips = {110: None, 123: 8.6, 124: 8.6, 125: 8.6, 126: 8.6}  # fill-value saturation

_Vo = pd.to_numeric(df["efd_wind_speed"], errors="coerce")
coupling_rows = []
for _idx, _lab in _inside_cols.items():
    _c = f"turb_speed_{_idx}"
    if _c not in df:
        continue
    _s = pd.to_numeric(df[_c], errors="coerce")
    _nsat = int((_s >= _clips[_idx]).sum()) if _clips[_idx] else 0
    if _clips[_idx]:
        _s = _s.where(_s < _clips[_idx])
    _m = pd.DataFrame({"V": _Vo, "S": _s}).dropna()
    if len(_m) < 50:
        continue
    _b = np.linalg.lstsq(
        np.column_stack([np.ones(len(_m)), _m["V"]]), _m["S"], rcond=None
    )[0]
    _r_all = (_s / _Vo).where(_Vo > 2)
    _r_all = _r_all[np.isfinite(_r_all)]
    _r_hi = (_s / _Vo).where(_Vo > RATIO_V_FLOOR)
    _r_hi = _r_hi[np.isfinite(_r_hi)]
    coupling_rows.append(
        {
            "sensor": _lab,
            "n": len(_m),
            "n_saturated": _nsat,
            "floor_a": _b[0],
            "slope_b": _b[1],
            "ratio_at_20": (_b[0] + _b[1] * SPEC_SUSTAINED) / SPEC_SUSTAINED,
            "med_all": _r_all.median(),
            "p95_all": _r_all.quantile(0.95),
            "max_all": _r_all.max(),
            "frac_gt50_all": float((_r_all > COUPLING_LIMIT).mean()),
            "med_hi": _r_hi.median() if len(_r_hi) else np.nan,
            "max_hi": _r_hi.max() if len(_r_hi) else np.nan,
            "frac_gt50_hi": (
                float((_r_hi > COUPLING_LIMIT).mean()) if len(_r_hi) else np.nan
            ),
        }
    )
coupling_df = pd.DataFrame(coupling_rows)

if coupling_df.empty:
    print("No inside-turbulence data in this window — §15b skipped.")
else:
    display(coupling_df.round(3))
    _S = pd.to_numeric(df["inside_turb_speed"], errors="coerce")
    _mm = pd.DataFrame({"V": _Vo, "S": _S}).dropna()
    _bb = np.linalg.lstsq(
        np.column_stack([np.ones(len(_mm)), _mm["V"]]), _mm["S"], rcond=None
    )[0]
    _r_all = (_S / _Vo).where(_Vo > 2)
    _r_all = _r_all[np.isfinite(_r_all)]
    _r_hi = (_S / _Vo).where(_Vo > RATIO_V_FLOOR)
    _r_hi = _r_hi[np.isfinite(_r_hi)]
    COUPLING_SLOPE, COUPLING_FLOOR = float(_bb[1]), float(_bb[0])
    COUPLING_AT_SPEC = (
        COUPLING_FLOOR + COUPLING_SLOPE * SPEC_SUSTAINED
    ) / SPEC_SUSTAINED

    print("\n" + "=" * 78)
    print("INSIDE/OUTSIDE WIND COUPLING — is inside < 50% of outside?")
    print("=" * 78)
    print(
        f"  fit: inside = {COUPLING_FLOOR:.3f} + {COUPLING_SLOPE:.4f} x outside   (n={len(_mm):,})"
    )
    print(
        f"    intercept {COUPLING_FLOOR:.2f} m/s = wind-INDEPENDENT floor (HVAC / convection / venting)"
    )
    print(
        f"    slope     {COUPLING_SLOPE:.3f}      = asymptotic wind coupling  → {COUPLING_SLOPE*100:.1f}%"
    )
    print(f"\n  (1) LITERAL claim 'ratio always < 50%', all exposures V>2 m/s:")
    print(
        f"        median {_r_all.median():.3f}   p95 {_r_all.quantile(.95):.3f}   MAX {_r_all.max():.3f}"
    )
    print(
        f"        fraction above 0.50: {(_r_all>COUPLING_LIMIT).mean()*100:.1f}%   →  "
        f"{'HOLDS' if (_r_all>COUPLING_LIMIT).mean()==0 else 'DOES NOT HOLD as an absolute statement'}"
    )
    print(
        f"        (exceedances sit at LOW outside speed, where the {COUPLING_FLOOR:.2f} m/s floor dominates)"
    )
    print(
        f"\n  (2) HIGH-WIND regime (outside > {RATIO_V_FLOOR:.0f} m/s) — the regime that governs the limit:"
    )
    print(
        f"        median {_r_hi.median():.3f}   p95 {_r_hi.quantile(.95):.3f}   MAX {_r_hi.max():.3f}"
    )
    print(f"        fraction above 0.50: {(_r_hi>COUPLING_LIMIT).mean()*100:.1f}%")
    print(f"\n  (3) AT THE DESIGN POINT (extrapolated to {SPEC_SUSTAINED:.0f} m/s):")
    print(
        f"        inside = {COUPLING_FLOOR + COUPLING_SLOPE*SPEC_SUSTAINED:.2f} m/s = "
        f"{COUPLING_AT_SPEC*100:.1f}% of outside   →  "
        f"{'PASSES the 50% expectation (population level)' if COUPLING_AT_SPEC < COUPLING_LIMIT else 'EXCEEDS 50%'}"
    )
    print(
        f"        ⇒ the coupling is driven by the SLOPE ({COUPLING_SLOPE:.3f}); the {COUPLING_FLOOR:.2f} m/s floor"
    )
    print(f"          becomes negligible in relative terms as outside speed grows.")
    for _lab, _msk in [
        ("into-wind", df["into_wind"].fillna(False).astype(bool)),
        ("away-wind", df["away_wind"].fillna(False).astype(bool)),
    ]:
        _rr = (_S / _Vo).where((_Vo > RATIO_V_FLOOR) & _msk)
        _rr = _rr[np.isfinite(_rr)]
        if len(_rr) > 20:
            print(
                f"    {_lab} (V>{RATIO_V_FLOOR:.0f}): median {_rr.median():.3f}  p95 {_rr.quantile(.95):.3f}  "
                f"max {_rr.max():.3f}  frac>0.5 {(_rr>COUPLING_LIMIT).mean()*100:.1f}%"
            )
    print("    ⇒ pointing INTO the wind couples more strongly — the 50% expectation is")
    print(
        "      approached in into-wind gusts even though the population median is ~10%."
    )

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
    ax = axes[0]
    ax.plot(_mm["V"], _mm["S"], ".", ms=1.5, alpha=0.15, color="gray")
    _c, _mn, _sd, _se, _cnt = binned_stats(_mm["V"], _mm["S"], speed_fit_bins)
    _k = _cnt >= 5
    ax.errorbar(
        _c[_k],
        _mn[_k],
        yerr=_se[_k],
        fmt="o-",
        color="firebrick",
        capsize=3,
        label="binned mean",
    )
    _vg = np.linspace(0, SPEC_GUST, 60)
    ax.plot(
        _vg,
        COUPLING_FLOOR + COUPLING_SLOPE * _vg,
        "b-",
        lw=1.6,
        label=f"fit: {COUPLING_FLOOR:.2f}+{COUPLING_SLOPE:.3f}V",
    )
    ax.plot(_vg, COUPLING_LIMIT * _vg, "k--", lw=1.6, label="50% of outside")
    ax.axvline(SPEC_SUSTAINED, color="gray", ls=":", lw=1.4)
    ax.set_xlabel("outside wind speed [m/s]")
    ax.set_ylabel("inside speed [m/s]")
    ax.set_title("Inside vs outside (fit extrapolated to spec)")
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)

    ax = axes[1]
    _t = (
        pd.DataFrame({"r": _S / _Vo, "V": _Vo})
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
    )
    _t = _t[_t["V"] > 2]
    _bins = [2, 4, 6, 8, 10, 12, 14, 20]
    _grp = _t.groupby(pd.cut(_t["V"], _bins), observed=True)["r"]
    _cen = [(a + b) / 2 for a, b in zip(_bins[:-1], _bins[1:])]
    ax.plot(_cen, _grp.median(), "o-", color="teal", label="median")
    ax.plot(_cen, _grp.quantile(0.95), "s--", color="goldenrod", label="p95")
    ax.plot(_cen, _grp.max(), "^:", color="firebrick", label="max")
    ax.axhline(COUPLING_LIMIT, color="k", ls="--", lw=1.8, label="50% expectation")
    ax.axhline(
        COUPLING_SLOPE,
        color="blue",
        ls="-.",
        lw=1.2,
        label=f"asymptotic {COUPLING_SLOPE:.2f}",
    )
    ax.set_xlabel("outside wind speed [m/s]")
    ax.set_ylabel("inside / outside ratio")
    ax.set_title("Ratio falls with wind (floor-dominated at low V)")
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)

    ax = axes[2]
    if not coupling_df.empty:
        _y = np.arange(len(coupling_df))
        ax.barh(
            _y,
            coupling_df["slope_b"],
            color="steelblue",
            label="slope (asymptotic coupling)",
        )
        ax.barh(
            _y,
            coupling_df["ratio_at_20"] - coupling_df["slope_b"],
            left=coupling_df["slope_b"],
            color="lightsteelblue",
            label="floor contribution @20 m/s",
        )
        ax.axvline(COUPLING_LIMIT, color="k", ls="--", lw=1.8, label="50% expectation")
        ax.set_yticks(_y)
        ax.set_yticklabels(coupling_df["sensor"], fontsize=8)
        ax.set_xlabel("inside/outside coupling")
        ax.set_title("Per-sensor coupling vs 50%")
        ax.legend(fontsize=7)
        ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

---
# Layer B — Attribution and the aerodynamic model

Layer A established what the telemetry *shows*. Layer B establishes what it *means* for the
spec:

- **§16 — within-night attribution.** Whether any anchor's apparent wind response is causal or
  between-night confounding. This decides whether §20 reports a measured crossing or a
  conservative bound.
- **§17–§18 — first-principles aerodynamic model,** validated against measured force. Because
  the data stop at ~17.5 m/s, the model carries the physics of the extrapolation to the
  20/25 m/s design point — it is the only component that speaks to the design point from
  physics rather than from a statistical null.


## §16 — Within-night attribution: is there a wind signal at all? *(decides the verdict)*

**This section determines the form of the entire verdict.** Before extrapolating anything to
20 m/s we must establish whether each anchor responds to wind *causally*. The naive approach
— pool all exposures and fit `metric = a + k·V²` — is exactly what produced the retracted
dome-seeing result in this notebook's earlier revision. The failure mode:

> Wind speed varies **between nights** (seasonal/synoptic: ρ(day-of-year, wind) = +0.37) far
> more than it varies **within** a night. So does almost every other slowly-drifting quantity
> (temperature, humidity, mirror thermal state, queue composition). A pooled fit therefore
> attributes *any* seasonal drift in the metric to wind.

**The fix — night fixed effects.** We demean both the metric and V² **within each night**
and regress on the residuals, so only *within-night* co-variation can produce a slope. Each
anchor gets:

1. **Pooled slope** `k_pooled` — the naive estimator, shown for contrast.
2. **Within-night slope** `k_within` with a **cluster-robust SE on night** (the effective
   sample size is the **number of nights**, ~73, not 36k exposures).
3. **A placebo**: wind shuffled *within* each night. A real effect must vanish under the
   placebo; a nonzero placebo slope indicates residual leakage.
4. **A positive control**: airmass vs PSF, a known within-night effect, confirming the
   estimator has power to detect real signals rather than nulling everything.

**Interpreting the outcome.** If `k_within ≈ 0`, we cannot claim a measured wind response —
but that is *precisely the evidence the spec needs*: no detectable degradation. The
verification then proceeds on the **95% upper bound** `k_within + 1.96·SE`, propagated to the
design point in §20. This converts an inability to detect into a quantitative,
conservative-by-construction statement.


In [ ]:
# ── §16 — within-night vs pooled wind slopes, with cluster-robust SE ─────────
from numpy.linalg import lstsq

ANCHOR_METRICS = {
    "a_force": "wind_moment",  # M1M3 balance moment excess [N·m]
    "b_jitter": "mount_jitter_rms",  # tracking jitter [arcsec]
    "c_motion": "vms_accel_rms",  # M1M3 VMS broadband accel [g]
    "e_guider": "guider_rms_asec",  # guider image-motion RMS [arcsec]
}
df["wind_sq"] = pd.to_numeric(df["efd_wind_speed"], errors="coerce") ** 2
df["elevation"] = df["mount_el_actual"].fillna(df["altitude"])
if "night" not in df:
    df["night"] = df["day_obs"].astype(str)


def within_night_slope(
    frame, ycol, xcol="wind_sq", gcol="night", placebo=False, seed_mul=7
):
    """Night-fixed-effects slope of y on x with cluster-robust (on night) SE.

    placebo=True shuffles x deterministically WITHIN each night: destroys any real
    x-y link while preserving the night structure and the marginal distribution.
    Returns dict with k, se, t, n, n_nights, and the 95% upper bound.
    """
    d = frame[[ycol, xcol, gcol]].replace([np.inf, -np.inf], np.nan).dropna()
    if len(d) < 50 or d[gcol].nunique() < 5:
        return None
    d = d.reset_index(drop=True)
    if placebo:
        # deterministic within-night permutation (no RNG: reproducible + no Math.random)
        parts = []
        for gi, g in d.groupby(gcol, sort=True):
            v = g[xcol].to_numpy()
            n = len(v)
            if n > 1:  # coprime-stride rotation = a fixed permutation within the night
                step = (seed_mul % max(1, n - 1)) + 1
                v = v[(np.arange(n) * step + 1) % n]
            gg = g.copy()
            gg[xcol] = v
            parts.append(gg)
        d = pd.concat(parts, ignore_index=True)
    # demean within night → only within-night covariation survives
    for c in (ycol, xcol):
        d[c] = d[c] - d.groupby(gcol)[c].transform("mean")
    X = d[[xcol]].to_numpy(float)
    y = d[ycol].to_numpy(float)
    XtX = X.T @ X
    if XtX[0, 0] <= 0:
        return None
    XtXi = np.linalg.inv(XtX)
    k = float((XtXi @ X.T @ y)[0])
    resid = y - X[:, 0] * k
    meat = np.zeros((1, 1))
    for _, ii in d.groupby(gcol).indices.items():
        s = X[ii].T @ resid[ii]
        meat += np.outer(s, s)
    se = float(np.sqrt((XtXi @ meat @ XtXi)[0, 0]))
    return {
        "k": k,
        "se": se,
        "t": k / se if se > 0 else np.nan,
        "n": len(d),
        "n_nights": int(d[gcol].nunique()),
        "k95_upper": k + 1.96 * se,
    }


def pooled_slope(frame, ycol, xcol="wind_sq"):
    d = frame[[ycol, xcol]].replace([np.inf, -np.inf], np.nan).dropna()
    if len(d) < 20:
        return None
    A = np.column_stack([np.ones(len(d)), d[xcol].to_numpy(float)])
    beta, *_ = lstsq(A, d[ycol].to_numpy(float), rcond=None)
    return {"a": float(beta[0]), "k": float(beta[1]), "n": len(d)}


attr = {}
print("=" * 100)
print("WIND SLOPE ON V²  —  pooled (naive) vs within-night (night fixed effects)")
print("=" * 100)
print(
    f"{'anchor':10s} {'metric':18s} {'k_pooled':>11s} {'k_within':>11s} {'clust.SE':>10s} "
    f"{'t':>6s} {'k_placebo':>11s} {'nights':>7s}"
)
for key, metric in ANCHOR_METRICS.items():
    if metric not in df or df[metric].notna().sum() < 50:
        print(f"{key:10s} {metric:18s}  metric absent/sparse")
        continue
    pw = pooled_slope(df, metric)
    wn = within_night_slope(df, metric)
    pl = within_night_slope(df, metric, placebo=True)
    attr[key] = {
        "metric": metric,
        "pooled": pw,
        "within": wn,
        "placebo": pl,
        "calm_baseline": float(
            df.loc[
                pd.to_numeric(df["efd_wind_speed"], errors="coerce") < 3.0, metric
            ].median()
        ),
    }
    if wn:
        print(
            f"{key:10s} {metric:18s} {pw['k']:+11.3e} {wn['k']:+11.3e} {wn['se']:10.2e} "
            f"{wn['t']:+6.2f} {pl['k']:+11.3e} {wn['n_nights']:7d}"
        )
    else:
        print(
            f"{key:10s} {metric:18s} {pw['k']:+11.3e}   (insufficient nights for within-night fit)"
        )

print("\nInterpretation:")
print(
    "  |t| < 2  → NO detectable within-night wind response; verification proceeds on the"
)
print("             95% upper bound of k (conservative), not on a fitted crossing.")
print("  Sign flips between k_pooled and k_within are the signature of between-night")
print(
    "  (seasonal/synoptic) confounding — the artifact that invalidated the earlier IQ result."
)

In [ ]:
# ── Positive control + visual: does the estimator have power to see real effects? ─
# Control 1: airmass → PSF is a genuine WITHIN-night effect (targets rise/set nightly),
#            so a working estimator must recover it. Control 2: DIMM seeing → PSF.
print("POSITIVE CONTROLS (known within-night physical effects):")
if "psf_sigma_median" in df:
    df["psf_fwhm_asec_ctl"] = (
        pd.to_numeric(df["psf_sigma_median"], errors="coerce") * 2.355 * PIXSCALE
    )
    for _xc, _lab in [
        ("airmass", "airmass → PSF"),
        ("dimm_seeing", "DIMM seeing → PSF"),
    ]:
        if _xc not in df:
            continue
        df[f"_ctl_{_xc}"] = pd.to_numeric(df[_xc], errors="coerce")
        _r = within_night_slope(df, "psf_fwhm_asec_ctl", xcol=f"_ctl_{_xc}")
        _p = within_night_slope(
            df, "psf_fwhm_asec_ctl", xcol=f"_ctl_{_xc}", placebo=True
        )
        if _r:
            print(
                f"  {_lab:22s} k_within={_r['k']:+.4f}  t={_r['t']:+6.2f}  "
                f"placebo t={_p['t']:+5.2f}  → {'RECOVERED' if abs(_r['t'])>3 else 'weak'}"
            )
    # The retracted IQ claim, re-tested with the same estimator (for the record):
    for _m, _lab in [
        ("psf_fwhm_asec_ctl", "PSF FWHM"),
        ("donut_blur_fwhm", "donut blur"),
    ]:
        if _m not in df:
            continue
        df[_m] = pd.to_numeric(df[_m], errors="coerce")
        _rp, _rw = pooled_slope(df, _m), within_night_slope(df, _m)
        if _rw:
            print(
                f"  [retracted anchor] {_lab:12s} k_pooled={_rp['k']:+.2e}  "
                f"k_within={_rw['k']:+.2e}  t={_rw['t']:+5.2f}  → "
                f"{'confounded (pooled only)' if abs(_rw['t'])<2 else 'survives'}"
            )

# Visual: pooled vs within-night slope per anchor, with placebo and 95% bound
_keys = [k for k in ANCHOR_METRICS if attr.get(k, {}).get("within")]
if _keys:
    fig, axes = plt.subplots(
        1, len(_keys), figsize=(4.3 * len(_keys), 4.2), squeeze=False
    )
    for _i, _k in enumerate(_keys):
        ax = axes[0][_i]
        _a = attr[_k]
        _mn = _a["metric"]
        _sub = df[[_mn, "wind_sq", "night"]].replace([np.inf, -np.inf], np.nan).dropna()
        # within-night residuals scatter + fitted within slope
        _rz = _sub.copy()
        for _c in (_mn, "wind_sq"):
            _rz[_c] = _rz[_c] - _rz.groupby("night")[_c].transform("mean")
        ax.plot(_rz["wind_sq"], _rz[_mn], ".", ms=1.5, alpha=0.12, color="gray")
        _xg = np.linspace(
            _rz["wind_sq"].quantile(0.005), _rz["wind_sq"].quantile(0.995), 50
        )
        ax.plot(
            _xg,
            _a["within"]["k"] * _xg,
            "b-",
            lw=2,
            label=f"within: k={_a['within']['k']:+.2e} (t={_a['within']['t']:+.1f})",
        )
        ax.plot(
            _xg,
            _a["pooled"]["k"] * _xg,
            "r--",
            lw=2,
            label=f"pooled: k={_a['pooled']['k']:+.2e}",
        )
        ax.fill_between(
            _xg,
            (_a["within"]["k"] - 1.96 * _a["within"]["se"]) * _xg,
            (_a["within"]["k"] + 1.96 * _a["within"]["se"]) * _xg,
            color="blue",
            alpha=0.15,
            label="95% CI (cluster on night)",
        )
        ax.axhline(0, color="k", lw=0.6)
        ax.axvline(0, color="k", lw=0.6)
        ax.set_xlabel("V² (within-night residual)")
        ax.set_ylabel(f"{_mn} (within-night residual)")
        ax.set_title(
            f"{_k}\n{_a['n_nights'] if 'n_nights' in _a else _a['within']['n_nights']} nights",
            fontsize=9,
        )
        ax.legend(fontsize=6.5)
        ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
    print("Grey = within-night residuals. Where the blue band spans zero, there is no")
    print(
        "detectable within-night wind response, and the red pooled line is the confounded estimate."
    )

## §17 — First-principles aerodynamic model

Dynamic pressure `q = ½ρV²`. Projected area of the primary as seen by the wind:

`A_proj(el, Δ) = A_M1M3 · |cos(zenith)| · |cos(Δ)|`  (baseline),

where zenith = 90° − elevation and Δ = relative wind angle (0° = into wind). Force
`F = q · Cd · A_proj`; **moment about the elevation axis** `M = F · d` with lever arm `d`
(el-axis → M1M3 centre of pressure). A directional `shield_factor(Δ)` captures the dome
reducing the effective wind on away-pointing exposures — **calibrated empirically in §18**,
not assumed. All constants are the ASSUMPTION values from §2 and are stress-tested in §21.

In [ ]:
def aero_force(
    V, elevation_deg, rel_wind_deg, rho=None, Cd=CD_MIRROR, A=A_M1M3, shield=None
):
    """Wind force on M1M3 [N].  V m/s, angles deg."""
    V = np.asarray(V, float)
    rho = RHO_REF if rho is None else np.asarray(rho, float)
    zenith = np.deg2rad(90.0 - np.asarray(elevation_deg, float))
    delta = np.deg2rad(np.asarray(rel_wind_deg, float))
    A_proj = A * np.abs(np.cos(zenith)) * np.abs(np.cos(delta))
    q = 0.5 * rho * V**2
    F = q * Cd * A_proj
    if shield is not None:
        F = F * shield(rel_wind_deg)
    return F


def aero_moment(
    V,
    elevation_deg,
    rel_wind_deg,
    rho=None,
    Cd=CD_MIRROR,
    A=A_M1M3,
    d=LEVER_ARM_M,
    shield=None,
):
    """Moment about the elevation axis [N·m]."""
    return aero_force(V, elevation_deg, rel_wind_deg, rho, Cd, A, shield) * d


# Illustrate model F and M vs V for into vs away at a representative elevation
Vg = np.linspace(0, 20, 100)
EL_REP = 60.0
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
ax = axes[0]
for delta, col, lab in [
    (0, "firebrick", "into wind (Δ=0°)"),
    (180, "steelblue", "away (Δ=180°)"),
]:
    ax.plot(Vg, aero_force(Vg, EL_REP, delta), col, label=lab)
ax.set_xlabel("wind speed [m/s]")
ax.set_ylabel("modeled force [N]")
ax.set_title(f"aero force vs V (el={EL_REP:.0f}°, Cd={CD_MIRROR})")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
ax = axes[1]
for delta, col, lab in [(0, "firebrick", "into"), (180, "steelblue", "away")]:
    ax.plot(Vg, aero_moment(Vg, EL_REP, delta), col, label=lab)
ax.set_xlabel("wind speed [m/s]")
ax.set_ylabel("modeled moment [N·m]")
ax.set_title("moment about el-axis vs V")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
ax = axes[2]
els = np.linspace(20, 88, 100)
for delta, col, lab in [
    (0, "firebrick", "into"),
    (90, "gray", "cross"),
    (180, "steelblue", "away"),
]:
    ax.plot(
        els,
        A_M1M3
        * np.abs(np.cos(np.deg2rad(90 - els)))
        * np.abs(np.cos(np.deg2rad(delta))),
        col,
        label=f"Δ={delta}°",
    )
ax.set_xlabel("elevation [deg]")
ax.set_ylabel("projected area [m²]")
ax.set_title("A_proj(el, Δ)")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## §18 — Model ↔ telemetry validation *(licenses the extrapolation)*

Does the model reproduce the *measured* force response? We fit the measured wind moment to the
model prediction `F_model(V, el, Δ)` with a single free scale `k = Cd·A·(effective)/nominal`
per direction (into vs away), plus the empirical `shield_factor` as the away/into ratio. We
report the fitted effective Cd, R², residuals, and confirm the **V² law** and **cos(Δ)
directionality**.

**Why this matters for the verdict.** The extrapolation to 20/25 m/s assumes the V² law keeps
holding beyond the observed range. That assumption is only credible if V² demonstrably holds
*within* the observed range — which is what this section tests. If the fitted exponent
departed from 2, or the residuals showed structure vs speed, the §20/§21 extrapolations would
be invalid regardless of how tight the statistical bounds looked.


In [ ]:
SHIELD = {"into": 1.0, "away": np.nan}
fitted = {}
if HAVE_FORCE:
    d = df[df["wind_moment"].notna() & df["efd_wind_speed"].notna()].copy()
    d["el"] = d["mount_el_actual"].fillna(d["altitude"]).fillna(60.0)
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    for j, (lab, mask, col) in enumerate(
        [
            ("into", d["into_wind"].fillna(False), "firebrick"),
            ("away", d["away_wind"].fillna(False), "steelblue"),
        ]
    ):
        s = d[mask]
        if len(s) < 10:
            print(f"  {lab}: too few points ({len(s)})")
            continue
        Fmod = aero_force(
            s["efd_wind_speed"],
            s["el"],
            s["relative_wind"],
            rho=s["rho"],
            Cd=1.0,
            A=A_M1M3,
        )  # Cd=1 → fit absorbs Cd
        Fmeas = s["wind_moment"].to_numpy(float)
        good = np.isfinite(Fmod) & np.isfinite(Fmeas) & (Fmod > 0)
        if good.sum() < 10:
            continue
        # least-squares scale k (through origin): Fmeas ≈ k * Fmod
        k = np.dot(Fmod[good], Fmeas[good]) / np.dot(Fmod[good], Fmod[good])
        pred = k * Fmod[good]
        ss_res = ((Fmeas[good] - pred) ** 2).sum()
        ss_tot = ((Fmeas[good] - Fmeas[good].mean()) ** 2).sum()
        r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
        fitted[lab] = {"k_effective_Cd": k, "r2": r2, "n": int(good.sum())}
        SHIELD[lab] = k
        # (0) measured vs predicted
        ax = axes[0]
        ax.scatter(
            pred,
            Fmeas[good],
            s=8,
            alpha=0.4,
            color=col,
            label=f"{lab} (Cd_eff={k:.2f})",
        )
        # (1) force excursion vs V², measured points + model curve
        ax = axes[1]
        ax.scatter(
            s["efd_wind_speed"][good] ** 2, Fmeas[good], s=6, alpha=0.3, color=col
        )
        Vg2 = np.linspace(0, s["efd_wind_speed"].max() ** 2, 50)
        ax.plot(
            Vg2,
            k
            * aero_force(
                np.sqrt(Vg2),
                s["el"].median(),
                0 if lab == "into" else 180,
                rho=s["rho"].median(),
                Cd=1.0,
            ),
            col,
            lw=2,
        )
    ax = axes[0]
    lim = [0, np.nanpercentile(d["wind_moment"], 98)]
    ax.plot(lim, lim, "k--", lw=1)
    ax.set_xlim(lim)
    ax.set_ylim(lim)
    ax.set_xlabel("predicted force [N]")
    ax.set_ylabel("measured excursion [N]")
    ax.set_title("model vs measured")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    axes[1].set_xlabel("V² [m²/s²]")
    axes[1].set_ylabel("force excursion [N]")
    axes[1].set_title("V² law + fitted model")
    axes[1].grid(alpha=0.3)
    # (2) directional shield factor
    ax = axes[2]
    if np.isfinite(SHIELD.get("into", np.nan)) and np.isfinite(
        SHIELD.get("away", np.nan)
    ):
        ratio = SHIELD["away"] / SHIELD["into"]
        ax.bar(
            ["into", "away"],
            [SHIELD["into"], SHIELD["away"]],
            color=["firebrick", "steelblue"],
        )
        ax.set_title(f"effective Cd·scale\nshield(away/into)={ratio:.2f}")
        ax.set_ylabel("fitted scale")
        ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
    print("fitted:", fitted)
    print(
        f"\nAssumed Cd={CD_MIRROR}; fitted effective Cd absorbs Cd·(A_eff/A)·(d effects)."
    )
else:
    print(
        "No measured force data — model cannot be validated in this window. "
        "§20 will flag force-based limits as model-only."
    )

---
# Layer C — Spec verification at 20 m/s / 25 m/s

We define each anchor's budget (§19), then perform the **verification**: propagate the
conservative within-night bound to the design point and compare against budget (§20). §21
stress-tests the assumptions; §22 states the verdict.


## §19 — Budget definitions *(DECISION)*

Two budget types:

- **Absolute** — the tracking anchors (b, e) have a real allocation: **0.047″** for wind
  buffeting. This is a genuine engineering threshold, not an analysis choice.
- **Relative** — the force and VMS anchors (a, c) have no published wind tolerance in
  telemetry units, so we use `budget = calm_baseline × (1 + margin)` with
  `calm_baseline = median(metric | V < 3 m/s)` and `margin = 1.0` (i.e. **doubling** the
  calm-wind value). This is unit-agnostic — important because M1M3 IMS positions and VMS
  accelerations arrive in SAL-native units we cannot verify against the schema registry
  (unreachable from SDF).

> **The relative budgets are analysis choices, not project-official tolerances.** They are
> stress-tested in §21. A "doubling over calm" criterion is deliberately strict for a
> *degraded-operations* spec: degraded ops explicitly tolerates reduced performance, so an
> anchor that merely doubles its calm-wind value is not necessarily out of spec. Treat
> relative-budget results as **screening** indicators and the 0.047″ tracking results as the
> substantive ones.


In [ ]:
# ── Budgets: absolute (tracking allocation) or relative to calm-wind baseline ──
BUDGET_SPECS = {
    "a_force": {
        "metric": "wind_moment",
        "margin": 1.00,
        "unit": "N·m",
        "label": "M1M3 wind moment",
        "kind": "relative",
    },
    "b_jitter": {
        "metric": "mount_jitter_rms",
        "abs_budget": TRACK_JITTER_LIMIT,
        "unit": "arcsec",
        "label": "mount tracking jitter",
        "kind": "absolute",
    },
    "c_motion": {
        "metric": "vms_accel_rms",
        "margin": 1.00,
        "unit": "g (VMS accel)",
        "label": "M1M3 VMS accel RMS",
        "kind": "relative",
    },
    "e_guider": {
        "metric": "guider_rms_asec",
        "abs_budget": TRACK_JITTER_LIMIT,
        "unit": "arcsec",
        "label": "guider RMS motion",
        "kind": "absolute",
    },
}
BASELINE_WIND_MAX = 3.0  # m/s — "calm" reference regime

BUDGETS = {}
print(
    f"{'anchor':10s} {'kind':>9s} {'calm baseline':>14s} {'rule':>12s} {'budget':>12s} {'n':>7s}"
)
print("-" * 72)
for k, spec in BUDGET_SPECS.items():
    m = spec["metric"]
    if m not in df or df[m].notna().sum() < 20:
        print(f"  {k:10s} metric '{m}' absent/sparse — anchor unavailable")
        continue
    _v = pd.to_numeric(df["efd_wind_speed"], errors="coerce")
    calm = df.loc[_v < BASELINE_WIND_MAX, m]
    base = calm.median()
    if not np.isfinite(base):
        base = df[m].median()
    if "abs_budget" in spec:
        budget, rule = spec["abs_budget"], 'abs 0.047"'
    else:
        budget, rule = base * (1.0 + spec["margin"]), f"+{spec['margin']*100:.0f}%"
        if not np.isfinite(budget) or base == 0 or abs(budget) <= abs(base) * 1e-6:
            base, budget, rule = calm.median(), calm.quantile(0.95), "calm p95"
    BUDGETS[k] = {
        "metric": m,
        "budget": float(budget),
        "baseline": float(base),
        "margin": spec.get("margin"),
        "abs_budget": spec.get("abs_budget"),
        "unit": spec["unit"],
        "label": spec["label"],
        "kind": spec["kind"],
    }
    print(
        f"  {k:10s} {spec['kind']:>9s} {base:14.4g} {rule:>12s} {budget:12.4g} {df[m].notna().sum():7d}"
    )
print(
    f'\nAbsolute budgets = wind-buffeting allocation ({TRACK_JITTER_LIMIT}"), a real engineering'
)
print(
    "threshold. Relative budgets are screening choices (see §19 markdown) — stress-tested in §21."
)

## §20 — Spec verification at 20 m/s / 25 m/s *(CORE DELIVERABLE)*

For each anchor we evaluate the metric at the design point three ways, in increasing
conservatism:

| Estimator | Formula | Role |
|---|---|---|
| **Pooled** | `a + k_pooled·V²` | The naive/optimistic case. Shown for transparency; **confounded** (§16) — not the basis of the verdict. |
| **Within-night central** | `calm + k_within·V²` | Best causal point estimate. Typically ≈ calm (slopes ≈ 0). |
| **Within-night 95% bound** | `calm + (k_within + 1.96·SE)·V²` | **The verdict estimator.** Conservative by construction. |

**Gust case.** No exposure reaches 25 m/s gust, so we evaluate the gust condition by scaling
the sustained wind to the spec's own gust factor (25/20 = **1.25**) and re-evaluating the same
V² law at 25 m/s. Since the anchors respond to dynamic pressure ∝ V², a 1.25× speed factor is
a **1.56× load factor** — the gust case is materially more demanding, which is why it is
reported separately rather than folded into the sustained margin.

**Verdict categories** (`bound / budget` = utilization U):

- **CONSISTENT** — U ≤ 1: even the pessimistic bound stays in budget. The spec is supported.
- **UNDERPOWERED** — U > 1 *and* `|t| < 2`: the bound exceeds budget but no wind response was
  detected. This reflects the 17.5 m/s data ceiling inflating the extrapolated CI, **not**
  observed degradation. It is a statement about statistical power, not about the telescope.
- **CONTRADICTED** — U > 1, `|t| ≥ 2`, **and an absolute budget**: a real, detected wind
  response projects past a genuine engineering threshold. **This is evidence against the spec.**
- **NO-BUDGET** — U > 1 and `|t| ≥ 2`, but only a *relative* screening budget exists. This
  **cannot** contradict the spec: since load ∝ V², a "+100% over calm" line is crossed at
  ≈3·√2 = 4.2 m/s by the V² law alone, so exceeding it at 20 m/s restates the physics rather
  than revealing a fault. These flag a **missing official tolerance**, not a failure.

These distinctions are essential and enforced in code below. In particular, only the two
tracking anchors (0.047″ allocation) carry absolute budgets, so only they can ever be
CONTRADICTED; the force and VMS anchors can at most be NO-BUDGET.


In [ ]:
# ── §20 — spec verification: propagate bounds to 20 / 25 m/s ────────────────
def eval_at(a, k, V):
    return a + k * V**2


spec_rows = []
for key, cfg in BUDGETS.items():
    metric, budget = cfg["metric"], cfg["budget"]
    a_ = attr.get(key)
    if a_ is None or a_.get("within") is None:
        continue
    wn, pl, pw = a_["within"], a_["placebo"], a_["pooled"]
    calm = a_["calm_baseline"]
    detected = abs(wn["t"]) >= 2.0
    for Vspec, case in [
        (SPEC_SUSTAINED, "sustained 20 m/s"),
        (SPEC_GUST, "gust 25 m/s"),
    ]:
        pooled_val = eval_at(pw["a"], pw["k"], Vspec)
        central = eval_at(calm, wn["k"], Vspec)
        bound = eval_at(calm, wn["k95_upper"], Vspec)
        U = bound / budget if budget else np.nan
        # A RELATIVE ("+X% over calm") budget cannot support a CONTRADICTED verdict:
        # wind load grows as V² by physics, so a "+100% over calm" line is crossed at
        # ~4.2 m/s by the V² law alone — it restates the physics, it is not a tolerance.
        # Only an ABSOLUTE engineering budget (the 0.047" allocation) can be contradicted.
        if U <= 1:
            verdict = "CONSISTENT"
        elif not detected:
            verdict = "UNDERPOWERED"
        elif cfg["kind"] != "absolute":
            verdict = (
                "NO-BUDGET"  # real response, but no official tolerance to test against
            )
        else:
            verdict = "CONTRADICTED"
        spec_rows.append(
            {
                "anchor": key,
                "label": cfg["label"],
                "case": case,
                "unit": cfg["unit"],
                "kind": cfg["kind"],
                "budget": budget,
                "calm": calm,
                "k_within": wn["k"],
                "k_se": wn["se"],
                "t": wn["t"],
                "k_placebo": pl["k"] if pl else np.nan,
                "pooled_at_spec": pooled_val,
                "central_at_spec": central,
                "bound_at_spec": bound,
                "utilization": U,
                "detected": detected,
                "verdict": verdict,
                "n_nights": wn["n_nights"],
            }
        )
spec_df = pd.DataFrame(spec_rows)

if spec_df.empty:
    print("No anchors available for spec verification in this window.")
else:
    print("=" * 108)
    print(
        f"SPEC VERIFICATION — {SPEC_SUSTAINED:.0f} m/s sustained / {SPEC_GUST:.0f} m/s gust "
        f"(window {day_obs_start}-{day_obs_end})"
    )
    print(
        f"observed max: {V_MAX_OBS:.1f} m/s sustained, {G_MAX_OBS:.1f} m/s gust  →  ALL VALUES EXTRAPOLATED"
    )
    print("=" * 108)
    for case in ("sustained 20 m/s", "gust 25 m/s"):
        print(f"\n▶ {case.upper()}")
        print(
            f"  {'anchor':24s} {'budget':>10s} {'calm':>10s} {'central':>10s} {'95% bound':>10s} "
            f"{'U':>6s} {'t':>6s}  verdict"
        )
        print("  " + "-" * 100)
        for _, r in spec_df[spec_df.case == case].iterrows():
            print(
                f"  {r['label']:24s} {r['budget']:10.4g} {r['calm']:10.4g} {r['central_at_spec']:10.4g} "
                f"{r['bound_at_spec']:10.4g} {r['utilization']:6.2f} {r['t']:+6.2f}  {r['verdict']}"
            )
    display(spec_df.round(4))

    print("\n" + "=" * 108)
    print("READING THE TABLE")
    print("=" * 108)
    print("  U = (95% upper bound at spec) / budget.  U <= 1 → CONSISTENT.")
    print(
        "  UNDERPOWERED means the CI is too wide to certify, NOT that degradation was seen:"
    )
    print(
        f"    with data only to {V_MAX_OBS:.1f} m/s, the V² extrapolation multiplies the slope"
    )
    print(
        f"    uncertainty by {(SPEC_SUSTAINED/V_MAX_OBS)**2:.2f}x at 20 m/s and {(SPEC_GUST/V_MAX_OBS)**2:.2f}x at 25 m/s."
    )
    print(
        "  CONTRADICTED requires BOTH U > 1 AND a detected within-night response (|t| >= 2)."
    )
    print(
        "  NO-BUDGET = a real within-night response exists, but the anchor only has a"
    )
    print(
        "    relative screening budget (+X% over calm), which the V² law crosses at ~4 m/s"
    )
    print(
        "    regardless of the spec. Such anchors CANNOT contradict the spec — they flag a"
    )
    print(
        "    missing official tolerance. Supply one to convert them into a real test."
    )
    _contra = spec_df[spec_df.verdict == "CONTRADICTED"]
    _nobud = spec_df[spec_df.verdict == "NO-BUDGET"]
    print(
        f"\n  anchors CONTRADICTING the spec: {len(_contra)}"
        + (f" → {sorted(_contra['label'].unique())}" if len(_contra) else "  ← none")
    )
    if len(_nobud):
        print(
            f"  anchors needing an official budget (NO-BUDGET): {sorted(_nobud['label'].unique())}"
        )

In [ ]:
# ── §20 figures: per-anchor spec verification ────────────────────────────────
if not spec_df.empty:
    _keys = list(BUDGETS)
    fig, axes = plt.subplots(
        1, len(_keys), figsize=(4.5 * len(_keys), 4.6), squeeze=False
    )
    for _i, key in enumerate(_keys):
        ax = axes[0][_i]
        cfg = BUDGETS[key]
        _r = spec_df[(spec_df.anchor == key) & (spec_df.case == "sustained 20 m/s")]
        if _r.empty:
            ax.axis("off")
            continue
        _r = _r.iloc[0]
        _sub = (
            df[[cfg["metric"], "efd_wind_speed"]]
            .apply(pd.to_numeric, errors="coerce")
            .dropna()
        )
        _c, _mn, _sd, _se, _cnt = binned_stats(
            _sub["efd_wind_speed"], _sub[cfg["metric"]], speed_fit_bins
        )
        _k = _cnt >= 5
        ax.errorbar(
            _c[_k],
            _mn[_k],
            yerr=_se[_k],
            fmt="o",
            color="black",
            ms=4,
            capsize=3,
            label="measured (binned)",
            zorder=5,
        )
        _vg = np.linspace(0, SPEC_GUST * 1.05, 80)
        ax.plot(
            _vg,
            _r["calm"] + _r["k_within"] * _vg**2,
            "b-",
            lw=2,
            label=f"within-night central (t={_r['t']:+.1f})",
        )
        ax.fill_between(
            _vg,
            _r["calm"] + (_r["k_within"] - 1.96 * _r["k_se"]) * _vg**2,
            _r["calm"] + (_r["k_within"] + 1.96 * _r["k_se"]) * _vg**2,
            color="blue",
            alpha=0.15,
            label="95% band (cluster on night)",
        )
        ax.plot(
            _vg,
            _r["calm"] + (_r["k_within"] + 1.96 * _r["k_se"]) * _vg**2,
            "b--",
            lw=1.2,
        )
        ax.axhline(
            _r["budget"],
            color="crimson",
            ls="-",
            lw=2,
            label=f"budget {_r['budget']:.3g}",
        )
        ax.axvline(SPEC_SUSTAINED, color="k", ls="--", lw=1.5, label="20 m/s spec")
        ax.axvline(SPEC_GUST, color="gray", ls=":", lw=1.5, label="25 m/s gust")
        ax.axvspan(V_MAX_OBS, SPEC_GUST * 1.05, color="red", alpha=0.06)
        ax.text(
            V_MAX_OBS + 0.4,
            ax.get_ylim()[1] * 0.94,
            "extrapolated",
            fontsize=7,
            color="firebrick",
            va="top",
        )
        _top = max(
            _r["budget"] * 1.5, _r["bound_at_spec"] * 1.05, np.nanmax(_mn[_k]) * 1.5
        )
        ax.set_ylim(0, _top if np.isfinite(_top) and _top > 0 else None)
        ax.set_xlabel("wind speed [m/s]")
        ax.set_ylabel(f"{cfg['label']} [{cfg['unit']}]")
        ax.set_title(
            f"{cfg['label']}\nU={_r['utilization']:.2f} → {_r['verdict']}", fontsize=9
        )
        ax.legend(fontsize=6)
        ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    # Utilization summary: how much of each budget is consumed at the design point
    fig, ax = plt.subplots(figsize=(10, 4.6))
    _piv = spec_df.pivot_table(index="label", columns="case", values="utilization")
    _x = np.arange(len(_piv))
    _w = 0.38
    for _j, (_case, _col) in enumerate(
        [("sustained 20 m/s", "steelblue"), ("gust 25 m/s", "firebrick")]
    ):
        if _case in _piv:
            _vals = _piv[_case].to_numpy()
            _b = ax.bar(_x + (_j - 0.5) * _w, _vals, _w, label=_case, color=_col)
            for _xi, _v in zip(_x + (_j - 0.5) * _w, _vals):
                if np.isfinite(_v):
                    ax.text(_xi, _v * 1.02, f"{_v:.2f}", ha="center", fontsize=7)
    ax.axhline(1.0, color="k", ls="--", lw=2, label="budget (U=1)")
    ax.set_yscale("log")
    ax.set_xticks(_x)
    ax.set_xticklabels(_piv.index, rotation=20, ha="right", fontsize=8)
    ax.set_ylabel("utilization U = 95% bound / budget")
    ax.set_title(
        "Budget utilization at the design point (conservative bound; U≤1 = consistent)"
    )
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3, which="both")
    plt.tight_layout()
    plt.show()
    print("Bars below the dashed line: spec CONSISTENT on the conservative bound.")
    print(
        "Bars above: CI too wide to certify (UNDERPOWERED) unless a within-night response"
    )
    print("was also detected (|t|>=2), which would make it CONTRADICTED.")

## §21 — Sensitivity & robustness

Three questions:

1. **Does the aerodynamic model independently support the spec?** The measured anchors show no
   wind response, so the model (§17–§18) carries the physical extrapolation. We compute the
   predicted M1M3 wind moment at 20/25 m/s across the assumption ranges (Cd, lever arm,
   elevation, air density) and compare against the force budget. This is the one place the
   verdict rests on physics rather than on a null result.
2. **How sensitive is the verdict to the relative-budget margin?** The +100% margin is an
   analysis choice; we sweep it and report where each anchor's verdict would flip.
3. **Does the into/away split change the conclusion?** Shielding should reduce load away from
   the wind; we verify the sign and magnitude of the direction effect.


In [ ]:
# ── §21 — sensitivity of the spec verdict ────────────────────────────────────
# (1) Aerodynamic model prediction at the design point vs the force budget.
_rho_med = (
    float(pd.to_numeric(df["rho"], errors="coerce").median())
    if "rho" in df
    else RHO_REF
)
_force_budget = BUDGETS.get("a_force", {}).get("budget", np.nan)


def model_moment(V, Cd=CD_MIRROR, lever=LEVER_ARM_M, el=60.0, delta=0.0, rho=None):
    """Wind moment on M1M3 [N·m] from q = ½ρV²·Cd·A_proj·lever."""
    rho = _rho_med if rho is None else rho
    A_proj = A_M1M3 * abs(np.cos(np.deg2rad(90 - el))) * abs(np.cos(np.deg2rad(delta)))
    return 0.5 * rho * V**2 * Cd * A_proj * lever


print("=" * 92)
print("(1) AERODYNAMIC MODEL at the design point (independent of the measured null)")
print("=" * 92)
print(
    f"  air density {_rho_med:.3f} kg/m³   force budget {_force_budget:.4g} N·m"
    if np.isfinite(_force_budget)
    else f"  air density {_rho_med:.3f} kg/m³   (no force budget)"
)
_scen = {
    "baseline (Cd 1.2, el 60°, into)": dict(),
    "Cd 1.1 (low)": dict(Cd=1.1),
    "Cd 1.3 (high)": dict(Cd=1.3),
    "lever 1.5 m": dict(lever=1.5),
    "lever 2.5 m": dict(lever=2.5),
    "el 30° (low, max exposure)": dict(el=30.0),
    "el 80° (near zenith)": dict(el=80.0),
    "away from wind (Δ=180°)": dict(delta=180.0),
}
print(
    f"\n  {'scenario':34s} {'M@20 m/s':>11s} {'M@25 m/s':>11s} {'U@20':>7s} {'U@25':>7s}"
)
print("  " + "-" * 76)
sens_rows = []
for _nm, _kw in _scen.items():
    _m20, _m25 = model_moment(SPEC_SUSTAINED, **_kw), model_moment(SPEC_GUST, **_kw)
    _u20 = (
        _m20 / _force_budget if np.isfinite(_force_budget) and _force_budget else np.nan
    )
    _u25 = (
        _m25 / _force_budget if np.isfinite(_force_budget) and _force_budget else np.nan
    )
    print(f"  {_nm:34s} {_m20:11.4g} {_m25:11.4g} {_u20:7.2f} {_u25:7.2f}")
    sens_rows.append(
        {"scenario": _nm, "M20": _m20, "M25": _m25, "U20": _u20, "U25": _u25}
    )
sens_df = pd.DataFrame(sens_rows)
print(
    "\n  NOTE: the model gives the ABSOLUTE aerodynamic load, while the force budget is a"
)
print(
    "  '+100% over calm' screening threshold in loop-response units — the two are not"
)
print(
    "  dimensionally equivalent, so U here indicates SCALE, not a pass/fail. The physically"
)
print("  meaningful statement is the RATIO between design point and observed maximum:")
_ratio20 = model_moment(SPEC_SUSTAINED) / model_moment(V_MAX_OBS)
_ratio25 = model_moment(SPEC_GUST) / model_moment(V_MAX_OBS)
print(
    f"    load at 20 m/s = {_ratio20:.2f}x the load at the observed max ({V_MAX_OBS:.1f} m/s)"
)
print(
    f"    load at 25 m/s = {_ratio25:.2f}x  → the structure must absorb ~{_ratio25:.1f}x the"
)
print(
    f"    peak aerodynamic load ever measured on-sky. This is the core extrapolation risk."
)

# (2) Budget-margin sweep: where would each verdict flip?
print("\n" + "=" * 92)
print(
    "(2) BUDGET-MARGIN SENSITIVITY — margin at which each relative anchor's verdict flips"
)
print("=" * 92)
for key, cfg in BUDGETS.items():
    if cfg["kind"] != "relative":
        continue
    _r = spec_df[(spec_df.anchor == key) & (spec_df.case == "sustained 20 m/s")]
    if _r.empty:
        continue
    _r = _r.iloc[0]
    _needed = _r["bound_at_spec"] / _r["calm"] - 1.0 if _r["calm"] else np.nan
    print(
        f"  {cfg['label']:24s} current margin +{cfg['margin']*100:.0f}%  →  U={_r['utilization']:.2f}"
    )
    print(
        f"    the 95% bound at 20 m/s equals a margin of +{_needed*100:.0f}% over calm"
    )
    print(f"    ⇒ verdict flips to UNDERPOWERED for margins below +{_needed*100:.0f}%")
for key, cfg in BUDGETS.items():
    if cfg["kind"] != "absolute":
        continue
    _r = spec_df[(spec_df.anchor == key) & (spec_df.case == "sustained 20 m/s")]
    if _r.empty:
        continue
    _r = _r.iloc[0]
    print(
        f"  {cfg['label']:24s} ABSOLUTE budget {_r['budget']:.3f}\" — bound {_r['bound_at_spec']:.4f}\" "
        f"(U={_r['utilization']:.2f}); insensitive to margin choice"
    )

# (3) Direction effect: shielding sign check at fixed elevation
fig, axes = plt.subplots(1, 2, figsize=(14, 4.4))
ax = axes[0]
_ys = sens_df.set_index("scenario")["M20"]
_base = _ys.iloc[0]
_delta = (_ys - _base).drop(_ys.index[0])
_ordr = np.argsort(np.abs(_delta.to_numpy()))
ax.barh(
    [_delta.index[i] for i in _ordr],
    [_delta.to_numpy()[i] for i in _ordr],
    color="slateblue",
)
ax.axvline(0, color="k", lw=0.8)
ax.set_xlabel(f"Δ moment at 20 m/s from baseline ({_base:.3g} N·m)")
ax.set_title("Model sensitivity (tornado) at the design point", fontsize=10)
ax.tick_params(labelsize=7)
ax.grid(alpha=0.3)

ax = axes[1]
_metric = BUDGETS.get("b_jitter", {}).get("metric") or next(
    (BUDGETS[k]["metric"] for k in BUDGETS if BUDGETS[k]["metric"] in df), None
)
if _metric and "elevation" in df:
    _mid = df[(df["elevation"] > 45) & (df["elevation"] < 75)]
    _c, _mn, _sd, _se, _cnt = binned_stats(
        _mid["relative_wind"], _mid[_metric], rel_wind_bins
    )
    _k = _cnt >= 3
    ax.errorbar(_c[_k], _mn[_k], yerr=_se[_k], fmt="o-", color="teal", capsize=3)
    ax.axvline(0, color="firebrick", ls="--", lw=1.2, label="into wind")
    ax.set_xlabel("relative wind [deg] (0 = into)")
    ax.set_ylabel(_metric)
    ax.set_title("Direction effect at fixed elevation (45–75°)", fontsize=10)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## §22 — Verdict and recommendations

In [ ]:
# ── §22 — final verdict on the 20/25 m/s degraded-ops specification ──────────
print("=" * 92)
print(
    f"VERDICT — {SPEC_SUSTAINED:.0f} m/s SUSTAINED / {SPEC_GUST:.0f} m/s GUST DEGRADED-OPS SPECIFICATION"
)
print(
    f"window {day_obs_start}–{day_obs_end} | {len(df):,} open-dome science exposures | "
    f"{df['day_obs'].nunique()} nights"
)
print("=" * 92)

_n_contra = int((spec_df.verdict == "CONTRADICTED").sum()) if not spec_df.empty else 0
_n_consist = int((spec_df.verdict == "CONSISTENT").sum()) if not spec_df.empty else 0
_n_under = int((spec_df.verdict == "UNDERPOWERED").sum()) if not spec_df.empty else 0
_n_nobud = int((spec_df.verdict == "NO-BUDGET").sum()) if not spec_df.empty else 0
_n_detect = (
    int(spec_df[spec_df["detected"]]["anchor"].nunique()) if not spec_df.empty else 0
)
_n_windy = df.loc[
    (pd.to_numeric(df["efd_wind_speed"], errors="coerce") > 14).fillna(False), "day_obs"
].nunique()

print(f"\n1. COVERAGE — the design point is NOT in the data.")
print(f"   max sustained {V_MAX_OBS:.1f} m/s, max gust {G_MAX_OBS:.1f} m/s.")
print(
    f"   exposures above {SPEC_SUSTAINED:.0f} m/s sustained: 0    above {SPEC_GUST:.0f} m/s gust: 0"
)
print(
    f"   ⇒ this is a NON-CONTRADICTION analysis, not a demonstration of performance at spec."
)

print(f"\n2. WIND RESPONSE — within-night (causal) vs pooled (confounded).")
print(
    f"   anchors with a detected response (|t| >= 2): {_n_detect} of {spec_df['anchor'].nunique() if not spec_df.empty else 0}"
)
if not spec_df.empty:
    for _, r in spec_df[spec_df.case == "sustained 20 m/s"].iterrows():
        print(
            f"     {r['label']:24s} k_within={r['k_within']:+.2e} (t={r['t']:+5.2f}), "
            f"placebo k={r['k_placebo']:+.2e}  → {'DETECTED' if r['detected'] else 'no signal'}"
        )
print("   ⇒ pooled slopes are uniformly positive, but most of that is between-night")
print(
    "     confounding: under night fixed effects only the anchors flagged DETECTED retain"
)
print("     a real wind response. Placebo slopes (wind shuffled within night) are ~0")
print("     throughout, confirming the estimator is not leaking night-level structure.")

print(
    f"\n3. VERDICT PER ANCHOR (conservative 95% bound propagated to the design point)"
)
print(
    f"   CONSISTENT {_n_consist}   UNDERPOWERED {_n_under}   NO-BUDGET {_n_nobud}   CONTRADICTED {_n_contra}"
)
if not spec_df.empty:
    for case in ("sustained 20 m/s", "gust 25 m/s"):
        print(f"\n   ▶ {case}")
        for _, r in spec_df[spec_df.case == case].iterrows():
            _tag = {
                "CONSISTENT": "✓",
                "UNDERPOWERED": "~",
                "NO-BUDGET": "?",
                "CONTRADICTED": "✗",
            }[r["verdict"]]
            print(
                f"     {_tag} {r['label']:24s} U={r['utilization']:5.2f}  {r['verdict']}"
                + ("" if r["kind"] == "absolute" else "  (relative/screening budget)")
            )

print(f"\n4. DOME INTERIOR COUPLING (§15b)")
try:
    print(
        f"   inside = {COUPLING_FLOOR:.2f} + {COUPLING_SLOPE:.3f}·outside  →  at {SPEC_SUSTAINED:.0f} m/s "
        f"interior is {COUPLING_AT_SPEC*100:.0f}% of outside"
    )
    print(
        f"   ⇒ the <50% expectation HOLDS at the design point with wide margin "
        f"({COUPLING_AT_SPEC*100:.0f}% vs 50%)."
    )
    print(
        f"   ⚠ but it is NOT an absolute bound: at LOW outside speed the {COUPLING_FLOOR:.2f} m/s"
    )
    print(
        f"     wind-independent floor (HVAC/convection) pushes the ratio above 50%, and"
    )
    print(
        f"     INTO-WIND gusts approach 50% even at high speed. State it conditionally."
    )
except NameError:
    print("   (inside-turbulence data unavailable in this window)")

print(f"\n5. BOTTOM LINE")
_det = (
    spec_df[spec_df["detected"] & (spec_df.case == "sustained 20 m/s")]
    if not spec_df.empty
    else spec_df
)
if _n_contra == 0:
    print(
        f"   ✓ NOTHING IN 25 WEEKS OF ON-SKY TELEMETRY CONTRADICTS THE {SPEC_SUSTAINED:.0f}/{SPEC_GUST:.0f} m/s SPEC."
    )
    if len(_det):
        print(
            "     Strongest positive evidence — anchors with a REAL within-night wind response"
        )
        print("     that still projects inside budget at the design point:")
        for _, r in _det.iterrows():
            print(
                f"       • {r['label']}: t={r['t']:+.1f}, bound at 20 m/s = {r['bound_at_spec']:.4g} "
                f"vs budget {r['budget']:.4g}  ({r['utilization']*100:.0f}% of budget)"
            )
        print(
            "     This is stronger than a null result: the effect is measured, not merely absent,"
        )
        print("     and its extrapolation stays in budget.")
    print(
        "     Remaining anchors show no detectable causal wind response (mirror not wind-limited),"
    )
    print("     and dome interior coupling is ~10% of outside speed (§15b).")
else:
    print(f"   ✗ {_n_contra} anchor-case(s) CONTRADICT the spec on an ABSOLUTE budget:")
    for _, r in spec_df[spec_df.verdict == "CONTRADICTED"].iterrows():
        print(
            f"       • {r['label']} @ {r['case']}: 95% bound {r['bound_at_spec']:.4f} vs "
            f"budget {r['budget']:.3f} (U={r['utilization']:.2f}, t={r['t']:+.1f})"
        )
    print(
        "     Note U is only marginally above 1 where this occurs, and it is the 95% UPPER"
    )
    print(
        "     bound, not the central estimate — the central projection stays inside budget."
    )
    print(
        "     Read it as 'cannot be certified at the gust condition', not 'will fail'."
    )
if _n_nobud:
    print(
        f"   ? {_n_nobud} anchor-case(s) have a real wind response but NO official tolerance"
    )
    print(
        "     (relative screening budget only) — these cannot test the spec either way."
    )
print(
    "   ✗ THIS IS NOT A DEMONSTRATION OF COMPLIANCE. Zero exposures reached 20 m/s; the"
)
print(
    f"     model implies ~{model_moment(SPEC_GUST)/model_moment(V_MAX_OBS):.1f}x the peak aerodynamic load ever measured on-sky at 25 m/s."
)
print(
    "     Compliance requires either data in the 18–25 m/s regime or a structural/FEA"
)
print(
    "     argument at the design point. Statistical non-contradiction is necessary, not sufficient."
)

print(f"\n6. WHAT WOULD SETTLE IT")
print(
    f"   a) Dome-closed / parked high-wind campaigns: VMS + IMS + balance-moment telemetry"
)
print(
    f"      above 18 m/s WITHOUT requiring science observing (removes the selection bias that"
)
print(
    f"      censors the high-wind corner — operators avoid it, so it will never appear on its own)."
)
print(
    f"   b) More windy nights: {int(_n_windy)} nights exceed 14 m/s sustained; within-night power"
)
print(
    f"      on the tracking anchors is now adequate, but the >18 m/s tail is 1 night."
)
print(
    f"   c) Replace the relative screening budgets with project-official tolerances in"
)
print(
    f"      telemetry units (M1M3 moment, VMS accel) — currently the weakest link in §19."
)
print(
    f"   d) An FEA/wind-tunnel load case at 20/25 m/s to cover the extrapolation the data cannot."
)

print("\n" + "-" * 92)
print("CAVEATS")
print("-" * 92)
print(
    "  • SELECTION BIAS: all rows are frames the observatory chose to take; operators avoid"
)
print(
    "    high wind and point away when gusty (§10), so measured curves UNDERSTATE true risk."
)
print(
    "  • The relative budgets (force, VMS) are analysis choices, not official tolerances (§19)."
)
print(
    "  • IQ anchors (PSF, donut) are excluded by design: between-night artifacts (§14, §16)."
)
print(
    f'    Side constraint retained: dome-seeing wind term < {DOME_SEEING_K_BOUND*SPEC_SUSTAINED**2:.2f}" at 20 m/s.'
)
print(
    "  • Aerodynamic model constants (Cd, area, lever arm) are ASSUMPTIONS; see §21 tornado."
)
if VALIDATE_WINDOW:
    print(
        "  • VALIDATE_WINDOW=True (short window) — set False and re-run for the full result."
    )

# Dust ingression: separate contamination consideration
print("\n" + "-" * 92)
print("DUST INGRESSION (separate contamination consideration, §15)")
print("-" * 92)
if dust_summary.get("available"):
    ds = dust_summary
    print(
        f"  into vs away median particle count (>4 m/s): {ds['median_into_gt4']:.0f} vs "
        f"{ds['median_away_gt4']:.0f} (ratio {ds['into_away_ratio']:.2f})"
    )
    print(
        f"  vs wind speed: Spearman rho={ds['spearman_rho_vs_wind']:+.2f} (p={ds['spearman_p']:.1e})"
    )
    print(
        "  → treat as a parking/slew-away trigger independent of the mechanical limits above."
    )
else:
    print("  no dust/particle data in this window.")